# FastCCC analysis for human hippocampal neurogenesis dataset

Paper context: Human hippocampal neurogenesis in adulthood, ageing and Alzheimer disease (`s41586-026-10169-4`). The local h5ad shown by the user contains `SampleNumber`, `Group`, `Cluster`, `Neurogenic`, and `Latent_Time`.

This notebook analyzes FastCCC communication at two levels:

1. Per biological sample FastCCC, then group comparison across YA, HA, SA, PCI, and AD.
2. Neurogenic latent-time window analysis, using `Latent_Time` to inspect trajectory-related communication.

The paper GEO accession is `GSE268609`; the local filename in the screenshot is `GSE286609_final.h5ad`, so this notebook follows the local server path.


In [ ]:
# ============================================================
# 1. Imports and publication-ready plotting style
# ============================================================
import os
import gc
import re
import inspect
import warnings
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as st
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import seaborn as sns

import fastccc

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

# Colorblind-safe palettes recommended for Nature/Science/Cell-style figures.
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]
TOL_MUTED = [
    '#332288', '#88CCEE', '#44AA99', '#117733', '#999933',
    '#DDCC77', '#CC6677', '#882255', '#AA4499', '#BBBBBB'
]
# Bright clinical palette: vivid on screen, still readable in print.
GROUP_COLORS = {
    'YA': '#2F80ED',   # bright blue
    'HA': '#27AE60',   # clear green
    'SA': '#9B51E0',   # violet
    'PCI': '#F2C94C',  # warm yellow
    'AD': '#EB5757',   # coral red
}
CELLTYPE_COLORS = {
    'Neurogenic': '#D55E00',
    'Astrocyte': '#009E73',
    'CA_neuron': '#0072B2',
    'Microglia': '#CC79A7',
    'OPC': '#E69F00',
    'Oligodendrocyte': '#56B4E9',
    'Vascular': '#44AA99',
    'GABA_neuron': '#882255',
    'Neuron_other': '#999933',
    'Other': '#BBBBBB',
}
ROLE_COLORS = {'ligand': '#2F80ED', 'receptor': '#EB5757'}
FIG_SINGLE_WIDTH = 3.5
FIG_DOUBLE_WIDTH = 7.2
GROUP_LABEL_ALIASES = {'MCI': 'PCI', 'mci': 'PCI', 'Mci': 'PCI'}


def normalize_group_label(value):
    return GROUP_LABEL_ALIASES.get(str(value).strip(), str(value).strip())


def normalize_group_series(values):
    return pd.Series(values, dtype='object').map(normalize_group_label)


def configure_publication_style():
    """Apply a clean, journal-style matplotlib/seaborn theme."""
    sns.set_theme(style='ticks', context='paper', palette=OKABE_ITO)
    mpl.rcParams.update({
        'figure.dpi': 120,
        'savefig.dpi': 600,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.04,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'SimHei'],
        'font.size': 12,
        'axes.labelsize': 12,
        'axes.titlesize': 13,
        'axes.titleweight': 'bold',
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
        'legend.title_fontsize': 12,
        'axes.linewidth': 0.9,
        'xtick.major.width': 0.9,
        'ytick.major.width': 0.9,
        'xtick.major.size': 4.2,
        'ytick.major.size': 4.2,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'legend.frameon': False,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'svg.fonttype': 'none',
        'axes.unicode_minus': False,
        'image.cmap': 'viridis',
    })


configure_publication_style()


def save_pub_figure(fig, filename, formats=('pdf', 'svg'), dpi=600, close=False):
    """Save a figure in vector formats for publication."""
    filename = Path(filename)
    base = filename.with_suffix('') if filename.suffix else filename
    base.parent.mkdir(parents=True, exist_ok=True)
    saved = []
    for fmt in formats:
        fmt = fmt.lstrip('.').lower()
        out = base.with_suffix(f'.{fmt}')
        raster_dpi = dpi if fmt in {'png', 'tif', 'tiff'} else min(dpi, 300)
        fig.savefig(
            out,
            format=fmt,
            dpi=raster_dpi,
            bbox_inches='tight',
            pad_inches=0.04,
            facecolor='white',
            edgecolor='none',
        )
        saved.append(out)
    print('Saved:', ', '.join(str(p) for p in saved))
    if close:
        plt.close(fig)
    return saved


def clean_axes(ax, grid=False, grid_axis='y'):
    """Remove chart junk while keeping light guide lines when useful."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if grid:
        ax.grid(True, axis=grid_axis, color='#E6E6E6', linewidth=0.755)
    else:
        ax.grid(False)
    ax.set_axisbelow(True)
    return ax


def ordered_groups(values=None):
    present = {normalize_group_label(v) for v in values} if values is not None else None
    groups = [g for g in GROUP_ORDER if present is None or g in present]
    if values is not None:
        extras = sorted({normalize_group_label(v) for v in values} - set(groups))
        groups.extend(extras)
    return groups


def group_palette(groups=None):
    groups = ordered_groups(groups) if groups is not None else list(GROUP_COLORS)
    return {g: GROUP_COLORS.get(normalize_group_label(g), OKABE_ITO[i % len(OKABE_ITO)]) for i, g in enumerate(groups)}


def categorical_palette(labels, fixed=None):
    labels = [normalize_group_label(x) for x in labels]
    fixed = {normalize_group_label(k): v for k, v in (fixed or {}).items()}
    fallback = TOL_MUTED + OKABE_ITO + [mcolors.to_hex(c) for c in sns.color_palette('tab20', max(20, len(labels)))]
    palette = {}
    j = 0
    for label in labels:
        if label in fixed:
            palette[label] = fixed[label]
        else:
            palette[label] = fallback[j % len(fallback)]
            j += 1
    return palette


def scaled_sizes(values, min_size=45, max_size=260, vmin=None, vmax=None):
    arr = pd.to_numeric(pd.Series(values), errors='coerce').fillna(0).to_numpy(dtype=float)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.full(arr.shape, (min_size + max_size) / 2)
    lo = float(finite.min() if vmin is None else vmin)
    hi = float(finite.max() if vmax is None else vmax)
    if np.isclose(hi, lo):
        return np.full(arr.shape, (min_size + max_size) / 2)
    scaled = np.clip((arr - lo) / (hi - lo), 0, 1)
    return min_size + scaled * (max_size - min_size)


def add_size_legend(ax, values, title, min_size=45, max_size=260, loc='upper left', bbox_to_anchor=(1.02, 1.0)):
    vals = pd.to_numeric(pd.Series(values), errors='coerce').dropna().to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None
    lo, hi = float(vals.min()), float(vals.max())
    qs = np.unique(np.round(np.nanpercentile(vals, [25, 50, 90]), 1))
    handles = [
        ax.scatter([], [], s=scaled_sizes([q], min_size, max_size, vmin=lo, vmax=hi)[0],
                   facecolors='none', edgecolors='#303030', linewidths=0.6, label=f'{q:g}')
        for q in qs
    ]
    return ax.legend(handles=handles, title=title, loc=loc, bbox_to_anchor=bbox_to_anchor,
                     frameon=False, borderaxespad=0.4, handletextpad=0.8)


def wrap_text(value, width=42):
    return '\n'.join(textwrap.wrap(str(value), width=width, break_long_words=False))


def wrap_axis_labels(ax, axis='y', width=42):
    if axis == 'y':
        labels = [wrap_text(t.get_text(), width) for t in ax.get_yticklabels()]
        ax.set_yticklabels(labels)
    else:
        labels = [wrap_text(t.get_text(), width) for t in ax.get_xticklabels()]
        ax.set_xticklabels(labels)
    return ax


def safe_filename(label, max_len=90):
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(label)).strip('_')
    return safe[:max_len] or 'figure'


print('Python executable:', os.sys.executable)
print('fastccc path:', fastccc.__file__)
print('FastCCC statistical_analysis_method signature:')
print(inspect.signature(fastccc.statistical_analysis_method))


## Publication figure standard

The plotting cells below use a Nature/Science/Cell-style theme: Arial/Helvetica-like sans-serif fonts, colorblind-safe palettes, minimal spines, compact single/double-column sizing, and vector-first export (`.pdf`/`.svg`).


In [ ]:
# ============================================================
# 2. Paths and parameters
# ============================================================
from pathlib import Path

DATA = Path('/bigdata/disk1/HDD1-Data/aya/hippocampal/GSE268609_final.h5ad')
DB_PATH = Path('/bigdata/disk1/HDD1-Data/aya/FastCCC-main/db/CellChatv1.6.1')
OUT = Path('/bigdata/disk1/HDD1-Data/aya/hippocampal/fastccc_GSE268609_cpdbv4_ensembl')

TMP = OUT / 'tmp_inputs'
RESULTS = OUT / 'per_sample_results'
FIG_DIR = Path('/home/aya/图片/海马')  # UTF-8 figure output directory
TABLE_DIR = OUT / 'tables'
ENRICH_DIR = OUT / 'enrichment'
LATENT_DIR = OUT / 'latent_time_fastccc'

for p in [OUT, TMP, RESULTS, FIG_DIR, TABLE_DIR, ENRICH_DIR, LATENT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY = 'SampleNumber'
GROUP_KEY = 'Group'
CELLTYPE_KEY = 'Cluster'
NEUROGENIC_KEY = 'Neurogenic'
LATENT_TIME_KEY = 'Latent_Time'
CONVERT_TYPE = 'ensembl'

GROUP_ORDER = ['YA', 'HA', 'SA', 'PCI', 'AD']
GROUP_CONTRASTS = [('AD', 'HA'), ('PCI', 'HA'), ('SA', 'HA'), ('AD', 'SA'), ('YA', 'HA')]

MIN_CELLS_PER_SAMPLE = 150
MIN_CELLTYPES_PER_SAMPLE = 2
MAX_CELLS_PER_CELLTYPE = 800
PVAL_SIG_CUTOFF = 0.05

RUN_FASTCCC_PER_SAMPLE = True
RERUN_EXISTING = False
RUN_LATENT_WINDOW_FASTCCC = True
RERUN_LATENT_EXISTING = False

print('DATA =', DATA)
print('DB_PATH =', DB_PATH)
print('OUT =', OUT)
print('FIG_DIR =', FIG_DIR)
print('CONVERT_TYPE =', CONVERT_TYPE)

assert DATA.exists(), f'Missing h5ad file: {DATA}'
assert DB_PATH.exists(), f'Missing FastCCC database: {DB_PATH}'

In [ ]:
# ============================================================
# 3. Read and inspect data
# ============================================================
adata = sc.read_h5ad(DATA)
adata.obs_names_make_unique()
adata.var_names_make_unique()

print(adata)
print('\nobs columns:')
print(adata.obs.columns.tolist())
print('\nfirst 20 genes:')
print(adata.var_names[:20].tolist())

for key in [SAMPLE_KEY, GROUP_KEY, CELLTYPE_KEY]:
    assert key in adata.obs.columns, f'Missing obs column: {key}'

print('\nGroup counts:')
display(adata.obs[GROUP_KEY].astype(str).value_counts().rename_axis('Group').reset_index(name='n_cells'))

print('\nSample counts:')
sample_counts = adata.obs.groupby([SAMPLE_KEY, GROUP_KEY], observed=False).size().rename('n_cells').reset_index()
display(sample_counts.sort_values([GROUP_KEY, SAMPLE_KEY]))

X = adata.X
try:
    x_min = float(X.min())
    x_max = float(X.max())
except Exception:
    x_min = float(np.min(X))
    x_max = float(np.max(X))
print('Original X min =', x_min)
print('Original X max =', x_max)


In [ ]:
# ============================================================
# 4. Paper-informed cell labels
# ============================================================
def clean_text(x):
    return str(x).strip().lower().replace(' ', '').replace('_', '-').replace('.', '')

def is_neurogenic_value(x):
    s = clean_text(x)
    if s in {'true', '1', 'yes', 'y', 'neurogenic'}:
        return True
    return any(k in s for k in ['nsc', 'neuroblast', 'immature'])

def infer_hippo_major(celltype, neurogenic_value=None):
    s = clean_text(celltype)
    if neurogenic_value is not None and is_neurogenic_value(neurogenic_value):
        return 'Neurogenic'
    if any(k in s for k in ['nsc', 'neuroblast', 'immature']):
        return 'Neurogenic'
    if 'astro' in s:
        return 'Astrocyte'
    if any(k in s for k in ['ca-neuron', 'ca-neurons', 'ca1', 'ca2', 'ca3', 'ca4', 'mgc', 'granule']):
        return 'CA_neuron'
    if any(k in s for k in ['microglia', 'micro']):
        return 'Microglia'
    if 'opc' in s:
        return 'OPC'
    if 'oligo' in s or 'moli' in s:
        return 'Oligodendrocyte'
    if 'endo' in s or 'vascular' in s or 'pericyte' in s:
        return 'Vascular'
    if any(k in s for k in ['gaba', 'inhibitory']):
        return 'GABA_neuron'
    if any(k in s for k in ['excitatory', 'neuron']):
        return 'Neuron_other'
    return 'Other'

obs = adata.obs.copy()
obs['sample_id'] = obs[SAMPLE_KEY].astype(str)
obs['diagnosis_group'] = normalize_group_series(obs[GROUP_KEY].astype(str)).values
obs['cluster_raw'] = obs[CELLTYPE_KEY].astype(str)

if NEUROGENIC_KEY in obs.columns:
    obs['major_celltype'] = [infer_hippo_major(c, n) for c, n in zip(obs['cluster_raw'], obs[NEUROGENIC_KEY])]
else:
    obs['major_celltype'] = obs['cluster_raw'].map(lambda x: infer_hippo_major(x, None))

obs['ccc_celltype'] = obs['cluster_raw']
obs.loc[obs['major_celltype'].eq('Neurogenic'), 'ccc_celltype'] = 'Neurogenic'

adata.obs['sample_id'] = obs['sample_id']
adata.obs['diagnosis_group'] = obs['diagnosis_group']
adata.obs['major_celltype'] = obs['major_celltype']
adata.obs['ccc_celltype'] = obs['ccc_celltype'].astype(str)

celltype_map = obs[['cluster_raw', 'major_celltype', 'ccc_celltype']].drop_duplicates().sort_values(['major_celltype', 'cluster_raw'])
celltype_map.to_csv(TABLE_DIR / 'celltype_major_map.tsv', sep='\t', index=False)

print('Major cell type counts:')
display(obs['major_celltype'].value_counts().rename_axis('major_celltype').reset_index(name='n_cells'))
print('FastCCC ccc_celltype counts:')
display(obs['ccc_celltype'].value_counts().rename_axis('ccc_celltype').reset_index(name='n_cells').head(50))
display(celltype_map)


In [ ]:
# ============================================================
# 5. FastCCC helper functions
# ============================================================
FASTCCC_LR_COMBINATION = 'Geometric'
FASTCCC_STYLE = None


def normalize_for_fastccc(sub):
    sub = sub.copy()
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)
    return sub


def downsample_by_celltype(sub, celltype_key='ccc_celltype', max_cells_per_celltype=MAX_CELLS_PER_CELLTYPE, random_state=0):
    if max_cells_per_celltype is None:
        return sub.copy()
    keep_idx = []
    rng = np.random.default_rng(random_state)
    for ct, idx in sub.obs.groupby(celltype_key, observed=False).indices.items():
        idx = np.array(list(idx))
        if len(idx) > max_cells_per_celltype:
            idx = rng.choice(idx, size=max_cells_per_celltype, replace=False)
        keep_idx.extend(idx.tolist())
    return sub[sorted(keep_idx)].copy()


def run_fastccc_on_h5ad(input_path, save_dir, meta_key='ccc_celltype', rerun=False):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    strength_file = save_dir / 'interactions_strength.tsv'
    pval_file = save_dir / 'pvals.tsv'
    percent_file = save_dir / 'percents_analysis.tsv'

    if strength_file.exists() and pval_file.exists() and percent_file.exists() and not rerun:
        strength = pd.read_csv(strength_file, sep='\t', index_col=0)
        pvals = pd.read_csv(pval_file, sep='\t', index_col=0)
        percents = pd.read_csv(percent_file, sep='\t', index_col=0)
        return strength, pvals, percents, 'exists_skip'

    strength, pvals, percents = fastccc.statistical_analysis_method(
        str(DB_PATH),
        None,
        str(input_path),
        CONVERT_TYPE,
        single_unit_summary='Quantile_0.9',
        complex_aggregation='Minimum',
        LR_combination=FASTCCC_LR_COMBINATION,
        style=FASTCCC_STYLE,
        meta_key=meta_key,
        save_path=str(save_dir),
    )

    strength.to_csv(strength_file, sep='\t')
    pvals.to_csv(pval_file, sep='\t')
    percents.to_csv(percent_file, sep='\t')
    return strength, pvals, percents, 'ok'


In [ ]:
# ============================================================
# 6. Run FastCCC per biological sample
# ============================================================
sample_meta = adata.obs[['sample_id', 'diagnosis_group']].drop_duplicates().sort_values(['diagnosis_group', 'sample_id']).reset_index(drop=True)
run_records = []

if RUN_FASTCCC_PER_SAMPLE:
    for i, row in sample_meta.iterrows():
        sample_id = str(row['sample_id'])
        group = str(row['diagnosis_group'])
        sub = adata[adata.obs['sample_id'].astype(str).eq(sample_id)].copy()
        n_cells_raw = int(sub.n_obs)
        n_celltypes_raw = int(sub.obs['ccc_celltype'].nunique())
        print(f'[{i+1}/{len(sample_meta)}] sample={sample_id}, group={group}, cells={n_cells_raw}, celltypes={n_celltypes_raw}')
        if n_cells_raw < MIN_CELLS_PER_SAMPLE or n_celltypes_raw < MIN_CELLTYPES_PER_SAMPLE:
            run_records.append({'sample_id': sample_id, 'group': group, 'n_cells_raw': n_cells_raw, 'n_celltypes_raw': n_celltypes_raw, 'status': 'skip_too_small'})
            continue
        try:
            sub = downsample_by_celltype(sub, 'ccc_celltype', MAX_CELLS_PER_CELLTYPE, random_state=1000+i)
            sub = normalize_for_fastccc(sub)
            input_path = TMP / f'{sample_id}.h5ad'
            save_dir = RESULTS / sample_id
            sub.write_h5ad(input_path)
            strength, pvals, percents, status = run_fastccc_on_h5ad(input_path, save_dir, meta_key='ccc_celltype', rerun=RERUN_EXISTING)
            run_records.append({'sample_id': sample_id, 'group': group, 'n_cells_raw': n_cells_raw, 'n_celltypes_raw': n_celltypes_raw, 'n_cells_used': int(sub.n_obs), 'n_celltypes_used': int(sub.obs['ccc_celltype'].nunique()), 'n_cellpairs': int(strength.shape[0]), 'n_lr_pairs': int(strength.shape[1]), 'status': status, 'result_dir': str(save_dir), 'input_path': str(input_path)})
            del sub, strength, pvals, percents
            gc.collect()
        except Exception as e:
            print('FastCCC failed:', repr(e))
            run_records.append({'sample_id': sample_id, 'group': group, 'n_cells_raw': n_cells_raw, 'n_celltypes_raw': n_celltypes_raw, 'status': 'error', 'error': repr(e)})
else:
    print('RUN_FASTCCC_PER_SAMPLE is False. Loading existing run_summary if present.')

if run_records:
    run_summary = pd.DataFrame(run_records)
    run_summary.to_csv(OUT / 'run_summary.tsv', sep='\t', index=False)
else:
    run_summary = pd.read_csv(OUT / 'run_summary.tsv', sep='\t')

if 'group' in run_summary.columns:
    run_summary['group'] = normalize_group_series(run_summary['group']).values

display(run_summary)
ok_samples = run_summary.loc[run_summary['status'].isin(['ok', 'exists_skip']), 'sample_id'].astype(str).tolist()
print('Successful samples:', len(ok_samples))
print(ok_samples)


In [ ]:
# ============================================================
# 7. Extract significant FastCCC interactions into a long table
# ============================================================
def parse_cellpair(cellpair):
    s = str(cellpair)
    for sep in ['|', ' -> ', '->', '__', ',']:
        if sep in s:
            parts = [x.strip() for x in s.split(sep)]
            if len(parts) == 2:
                return parts[0], parts[1]
    return s, np.nan


def load_sample_fastccc(sample_id):
    sample_dir = RESULTS / str(sample_id)
    strength = pd.read_csv(sample_dir / 'interactions_strength.tsv', sep='\t', index_col=0)
    pvals = pd.read_csv(sample_dir / 'pvals.tsv', sep='\t', index_col=0)
    percents = pd.read_csv(sample_dir / 'percents_analysis.tsv', sep='\t', index_col=0)
    return strength, pvals, percents


def extract_significant_sample(sample_id, group, pval_cutoff=PVAL_SIG_CUTOFF):
    strength, pvals, percents = load_sample_fastccc(sample_id)
    row_idx, col_idx = np.where(pvals.values < pval_cutoff)
    if len(row_idx) == 0:
        return pd.DataFrame(columns=['sample_id','group','cellpair','sender','receiver','sender_major','receiver_major','lr_pair','strength','pval','percent_ok','feature_id'])

    cellpairs = strength.index.to_numpy()[row_idx]
    lr_pairs = strength.columns.to_numpy()[col_idx]
    df = pd.DataFrame({
        'sample_id': str(sample_id),
        'group': str(group),
        'cellpair': cellpairs,
        'lr_pair': lr_pairs,
        'strength': strength.values[row_idx, col_idx],
        'pval': pvals.values[row_idx, col_idx],
        'percent_ok': percents.values[row_idx, col_idx],
    })
    df[['sender', 'receiver']] = df['cellpair'].apply(lambda x: pd.Series(parse_cellpair(x)))
    df['sender_major'] = df['sender'].map(lambda x: infer_hippo_major(x, None))
    df['receiver_major'] = df['receiver'].map(lambda x: infer_hippo_major(x, None))
    df['feature_id'] = df['cellpair'].astype(str) + ' || ' + df['lr_pair'].astype(str)
    return df[['sample_id','group','cellpair','sender','receiver','sender_major','receiver_major','lr_pair','strength','pval','percent_ok','feature_id']]


long_files = []
for _, row in run_summary[run_summary['status'].isin(['ok', 'exists_skip'])].iterrows():
    sample_id = str(row['sample_id'])
    group = str(row['group'])
    out_file = OUT / f'{sample_id}_fastccc_long_sig.tsv'
    if out_file.exists() and not RERUN_EXISTING:
        long_files.append(out_file)
        continue
    print('Extract significant interactions:', sample_id)
    df = extract_significant_sample(sample_id, group)
    df.to_csv(out_file, sep='\t', index=False)
    long_files.append(out_file)

fastccc_long = pd.concat([pd.read_csv(f, sep='\t') for f in long_files], ignore_index=True) if long_files else pd.DataFrame()
fastccc_long.to_csv(OUT / 'fastccc_long_sig_only.tsv', sep='\t', index=False)
print('fastccc_long shape:', fastccc_long.shape)
display(fastccc_long.head(20))


In [ ]:
# ============================================================
# 8. Group-level communication pair summaries
# ============================================================
if fastccc_long.empty:
    pair_sample = pd.DataFrame()
    pair_group = pd.DataFrame()
else:
    pair_sample = fastccc_long.groupby(
        ['sample_id','group','cellpair','sender','receiver','sender_major','receiver_major'],
        as_index=False,
    ).agg(
        sig_lr_count=('lr_pair','nunique'),
        sum_strength=('strength','sum'),
        mean_strength=('strength','mean'),
        min_pval=('pval','min'),
    )
    pair_sample.to_csv(TABLE_DIR / 'pair_sample_summary.tsv', sep='\t', index=False)

    pair_group = pair_sample.groupby(
        ['group','cellpair','sender','receiver','sender_major','receiver_major'],
        as_index=False,
    ).agg(
        n_samples=('sample_id','nunique'),
        mean_sum_strength=('sum_strength','mean'),
        mean_sig_lr_count=('sig_lr_count','mean'),
        min_pval=('min_pval','min'),
    )
    pair_group.to_csv(TABLE_DIR / 'pair_group_summary.tsv', sep='\t', index=False)

display(pair_sample.head(20) if not pair_sample.empty else pd.DataFrame({'message':['No pair_sample results.']}))
display(pair_group.sort_values('mean_sum_strength', ascending=False).head(30) if not pair_group.empty else pd.DataFrame({'message':['No pair_group results.']}))

if not pair_group.empty:
    top_pairs = pair_group.sort_values('mean_sum_strength', ascending=False).head(35)['cellpair'].unique().tolist()
    plot_df = pair_group[pair_group['cellpair'].isin(top_pairs)].copy()
    plot_df['pair_label'] = plot_df['sender'].astype(str) + ' -> ' + plot_df['receiver'].astype(str)
    groups_present = ordered_groups(plot_df['group'])
    pair_order = plot_df.groupby('pair_label')['mean_sum_strength'].max().sort_values().index.tolist()
    x_map = {g: i for i, g in enumerate(groups_present)}
    y_map = {p: i for i, p in enumerate(pair_order)}
    plot_df['x'] = plot_df['group'].astype(str).map(x_map)
    plot_df['y'] = plot_df['pair_label'].map(y_map)

    fig_h = max(3.8, min(9.2, 1.0 + 0.20 * len(pair_order)))
    fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))
    norm = mpl.colors.Normalize(vmin=0, vmax=max(float(plot_df['mean_sum_strength'].max()), 1e-9))
    sizes = scaled_sizes(plot_df['mean_sig_lr_count'], min_size=45, max_size=260)
    sca = ax.scatter(
        plot_df['x'], plot_df['y'],
        s=sizes,
        c=plot_df['mean_sum_strength'],
        cmap='cividis',
        norm=norm,
        edgecolors='white',
        linewidths=1.4,
        alpha=0.95,
    )
    ax.set_xticks(range(len(groups_present)))
    ax.set_xticklabels(groups_present)
    ax.set_yticks(range(len(pair_order)))
    ax.set_yticklabels(pair_order)
    wrap_axis_labels(ax, 'y', width=36)
    ax.set_xlabel('Diagnosis group')
    ax.set_ylabel('Sender -> receiver')
    ax.set_title('Top group-level communication pairs', loc='left')
    ax.set_xlim(-0.5, len(groups_present) - 0.5)
    ax.set_ylim(-0.6, len(pair_order) - 0.4)
    clean_axes(ax, grid=True, grid_axis='x')

    cbar = fig.colorbar(sca, ax=ax, pad=0.012, fraction=0.035)
    cbar.set_label('Mean summed FastCCC strength')
    add_size_legend(ax, plot_df['mean_sig_lr_count'], 'Mean no. significant LRIs',
                    min_size=45, max_size=260, bbox_to_anchor=(1.18, 1.0))
    fig.subplots_adjust(right=0.82)
    save_pub_figure(fig, FIG_DIR / 'group_level_top_communication_pairs_dotplot')
    plt.show()


In [ ]:
# ============================================================
# 9. Group contrasts for communication pairs
# ============================================================
def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full(p.shape, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    idx = np.where(ok)[0]
    order = idx[np.argsort(p[ok])]
    ranked = p[order]
    q = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out[order] = np.clip(q, 0, 1)
    return out


def differential_by_group(feature_df, feature_col, value_col, group1, group2, min_samples=3):
    if feature_df is None or feature_df.empty:
        print(f'Skip {group1}_vs_{group2}: feature_df is empty')
        return pd.DataFrame()
    if feature_col not in feature_df.columns or value_col not in feature_df.columns:
        print(f'Skip {group1}_vs_{group2}: missing columns')
        return pd.DataFrame()

    meta = run_summary.loc[run_summary['status'].isin(['ok', 'exists_skip']), ['sample_id', 'group']].copy()
    meta['sample_id'] = meta['sample_id'].astype(str)
    meta['group'] = normalize_group_series(meta['group'].astype(str)).values
    samples1 = meta.loc[meta['group'].eq(group1), 'sample_id'].tolist()
    samples2 = meta.loc[meta['group'].eq(group2), 'sample_id'].tolist()
    if len(samples1) < min_samples or len(samples2) < min_samples:
        print(f'Skip {group1}_vs_{group2}: not enough samples ({len(samples1)} vs {len(samples2)})')
        return pd.DataFrame()

    tmp = feature_df[[feature_col, 'sample_id', value_col]].copy()
    tmp['sample_id'] = tmp['sample_id'].astype(str)
    tmp[value_col] = pd.to_numeric(tmp[value_col], errors='coerce').fillna(0)
    pivot = tmp.pivot_table(index=feature_col, columns='sample_id', values=value_col, aggfunc='sum', fill_value=0)
    if pivot.empty:
        print(f'Skip {group1}_vs_{group2}: pivot table is empty')
        return pd.DataFrame()
    for sid in samples1 + samples2:
        if sid not in pivot.columns:
            pivot[sid] = 0.0

    rows = []
    for feature_id, row in pivot.iterrows():
        x = row[samples1].astype(float).values
        y = row[samples2].astype(float).values
        if np.all(x == 0) and np.all(y == 0):
            continue
        try:
            stat, pval = st.mannwhitneyu(x, y, alternative='two-sided')
        except Exception:
            stat, pval = np.nan, np.nan
        mean1 = float(np.mean(x))
        mean2 = float(np.mean(y))
        log2fc = float(np.log2((mean1 + 1e-9) / (mean2 + 1e-9)))
        rows.append({
            feature_col: feature_id,
            'contrast': f'{group1}_vs_{group2}',
            'group_1': group1,
            'group_2': group2,
            'n_group_1': len(samples1),
            'n_group_2': len(samples2),
            'mean_group_1': mean1,
            'mean_group_2': mean2,
            'log2fc_group1_vs_group2': log2fc,
            'pval': float(pval),
            'stat': float(stat) if np.isfinite(stat) else np.nan,
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out['fdr'] = bh_fdr(out['pval'].values)
        out['direction'] = np.where(out['log2fc_group1_vs_group2'] > 0, f'higher_in_{group1}', f'higher_in_{group2}')
    return out


pair_diff = pd.DataFrame()
if not pair_sample.empty:
    pair_diff_list = []
    for g1, g2 in GROUP_CONTRASTS:
        pair_diff_list.append(differential_by_group(pair_sample, 'cellpair', 'sum_strength', g1, g2))
    pair_diff = pd.concat([x for x in pair_diff_list if x is not None and not x.empty], ignore_index=True) if pair_diff_list else pd.DataFrame()
    if not pair_diff.empty:
        pair_info = pair_sample[['cellpair','sender','receiver','sender_major','receiver_major']].drop_duplicates('cellpair')
        pair_diff = pair_diff.merge(pair_info, on='cellpair', how='left')
        pair_diff.to_csv(TABLE_DIR / 'communication_pair_group_contrasts.tsv', sep='\t', index=False)
else:
    print('[WARNING] pair_sample is empty -- skipping group contrasts for communication pairs.')

display(pair_diff.sort_values(['fdr','pval']).head(40) if not pair_diff.empty else pd.DataFrame({'message':['No communication-pair contrast results.']}))

if not pair_diff.empty:
    top_plot = pair_diff.sort_values('pval').head(45).copy()
    top_plot['label'] = top_plot['sender'].astype(str) + ' -> ' + top_plot['receiver'].astype(str)
    top_plot['neglog10_p'] = -np.log10(top_plot['pval'].clip(lower=1e-300))
    label_order = top_plot.groupby('label')['neglog10_p'].max().sort_values().index.tolist()
    top_plot['label'] = pd.Categorical(top_plot['label'], categories=label_order, ordered=True)
    contrast_palette = categorical_palette(top_plot['contrast'].dropna().unique())

    fig_h = max(3.8, min(9.2, 1.0 + 0.20 * len(label_order)))
    fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))
    sns.scatterplot(
        data=top_plot,
        x='log2fc_group1_vs_group2',
        y='label',
        hue='contrast',
        size='neglog10_p',
        sizes=(24, 185),
        palette=contrast_palette,
        edgecolor='white',
        linewidth=0.655,
        alpha=0.95,
        ax=ax,
        legend='brief',
    )
    ax.axvline(0, color='#202020', lw=1.05, zorder=0)
    ax.set_title('Differential communication pairs between diagnosis groups', loc='left')
    ax.set_xlabel('log2 fold change of summed FastCCC strength')
    ax.set_ylabel('Sender -> receiver')
    wrap_axis_labels(ax, 'y', width=38)
    clean_axes(ax, grid=True, grid_axis='x')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, borderaxespad=0.4)
    fig.subplots_adjust(right=0.78)
    save_pub_figure(fig, FIG_DIR / 'communication_pair_group_contrasts_top45')
    plt.show()


In [ ]:
# ==================== 改进版绘图 ====================
if not pair_diff.empty:
    # 取 top 45 显著通信对
    top_plot = pair_diff.sort_values('pval').head(45).copy()
    top_plot['label'] = top_plot['sender'].astype(str) + ' -> ' + top_plot['receiver'].astype(str)
    top_plot['neglog10_p'] = -np.log10(top_plot['pval'].clip(lower=1e-300))

    # 按最大显著性排序 y 轴
    label_order = top_plot.groupby('label')['neglog10_p'].max().sort_values().index.tolist()
    top_plot['label'] = pd.Categorical(top_plot['label'], categories=label_order, ordered=True)

    contrast_palette = categorical_palette(top_plot['contrast'].dropna().unique())

    # 设置图高自适应
    fig_h = max(4, min(10, 1.0 + 0.22 * len(label_order)))
    fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))

    # 绘制散点图
    sns.scatterplot(
        data=top_plot,
        x='log2fc_group1_vs_group2',
        y='label',
        hue='contrast',
        size='neglog10_p',
        sizes=(24, 185),
        palette=contrast_palette,
        edgecolor='white',
        linewidth=0.655,
        alpha=0.95,
        ax=ax,
        legend='brief',
    )

    # 绘制纵向参考线
    ax.axvline(0, color='#202020', lw=1.05, zorder=0)

    # 标题和轴标签
    ax.set_title('Differential communication pairs between diagnosis groups', loc='left', fontsize=16, weight='bold')
    ax.set_xlabel('log2 fold change of summed FastCCC strength', fontsize=14)
    ax.set_ylabel('Sender -> Receiver', fontsize=14)

    # 调整 y 轴标签字体大小
    wrap_axis_labels(ax, 'y', width=38)  # 换行
    ax.tick_params(axis='y', labelsize=12)  # 纵轴字体
    ax.tick_params(axis='x', labelsize=12)  # 横轴字体

    # 清理网格和边框
    clean_axes(ax, grid=True, grid_axis='x')

    # 图例美化
    leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, borderaxespad=0.4)
    legend_handles = getattr(leg, 'legend_handles', getattr(leg, 'legendHandles', []))
    for lh in legend_handles:
        lh.set_alpha(1)  # 图例点透明度统一
        lh.set_edgecolor('black')  # 图例边框

    fig.subplots_adjust(right=0.78)  # 给图例留空间

    save_pub_figure(fig, FIG_DIR / 'communication_pair_group_contrasts_top45')
    plt.show()

In [ ]:
# ============================================================
# 10. Map FastCCC interaction IDs to ligand and receptor names
# ============================================================
def read_db_table(db_path, name):
    path = Path(db_path) / name
    if path.exists():
        return pd.read_csv(path)
    return None


def first_existing_col(df, candidates):
    if df is None:
        return None
    cols = list(df.columns)
    low = {c.lower(): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        if c.lower() in low:
            return low[c.lower()]
    return None


def clean_lr_name(x):
    s = str(x)
    s = s.replace('_HUMAN', '')
    s = re.sub(r'^protein_', '', s)
    s = re.sub(r'\s+', '', s)
    return s


def build_interaction_gene_map(db_path):
    db_path = Path(db_path)
    interaction = read_db_table(db_path, 'interaction_table.csv')
    assert interaction is not None, 'interaction_table.csv was not found in DB_PATH.'

    id_col = first_existing_col(interaction, ['id_cp_interaction', 'id_interaction', 'interaction_name'])
    lig_col = first_existing_col(interaction, ['multidata_1_id', 'partner_a', 'ligand', 'ligand_id'])
    rec_col = first_existing_col(interaction, ['multidata_2_id', 'partner_b', 'receptor', 'receptor_id'])
    print('interaction id column:', id_col)
    print('ligand id column:', lig_col)
    print('receptor id column:', rec_col)
    assert id_col is not None and lig_col is not None and rec_col is not None

    id_to_name = {}
    multidata = read_db_table(db_path, 'multidata_table.csv')
    if multidata is not None:
        mid = first_existing_col(multidata, ['id_multidata', 'multidata_id'])
        mname = first_existing_col(multidata, ['name', 'gene_name', 'hgnc_symbol'])
        if mid and mname:
            id_to_name.update(dict(zip(multidata[mid].astype(str), multidata[mname].astype(str))))

    protein = read_db_table(db_path, 'protein_table.csv')
    if protein is not None:
        pid = first_existing_col(protein, ['protein_multidata_id', 'id_protein'])
        pname = first_existing_col(protein, ['protein_name', 'name'])
        if pid and pname:
            id_to_name.update({str(k): str(v) for k, v in zip(protein[pid], protein[pname]) if str(k) not in id_to_name})

    gene = read_db_table(db_path, 'gene_table.csv')
    if gene is not None:
        gid = first_existing_col(gene, ['protein_id', 'id_gene'])
        gname = first_existing_col(gene, ['gene_name', 'hgnc_symbol'])
        if gid and gname:
            id_to_name.update({str(k): str(v) for k, v in zip(gene[gid], gene[gname]) if str(k) not in id_to_name})

    comp = read_db_table(db_path, 'complex_composition_table.csv')
    if comp is not None:
        cid = first_existing_col(comp, ['complex_multidata_id', 'complex_id'])
        pid = first_existing_col(comp, ['protein_multidata_id', 'protein_id'])
        if cid and pid:
            comp_names = {}
            for complex_id, part in comp[[cid, pid]].dropna().astype(str).values:
                comp_names.setdefault(complex_id, []).append(clean_lr_name(id_to_name.get(part, part)))
            for complex_id, names in comp_names.items():
                if complex_id not in id_to_name and names:
                    id_to_name[complex_id] = '_'.join(sorted(set(names)))

    out = interaction[[id_col, lig_col, rec_col]].drop_duplicates().rename(columns={id_col:'lr_pair', lig_col:'ligand_raw', rec_col:'receptor_raw'})
    out['ligand_gene'] = out['ligand_raw'].astype(str).map(lambda x: clean_lr_name(id_to_name.get(x, x)))
    out['receptor_gene'] = out['receptor_raw'].astype(str).map(lambda x: clean_lr_name(id_to_name.get(x, x)))
    out['lr_label'] = out['ligand_gene'].astype(str) + ' -> ' + out['receptor_gene'].astype(str)
    return out

interaction_map = build_interaction_gene_map(DB_PATH)
interaction_map.to_csv(TABLE_DIR / 'interaction_id_to_ligand_receptor.tsv', sep='\t', index=False)
print('interaction_map shape:', interaction_map.shape)
display(interaction_map.head(30))


In [ ]:
# ============================================================
# 11. LR-level group contrasts inside sender-receiver pairs
# ============================================================
if fastccc_long.empty:
    lr_sample = pd.DataFrame()
    lr_diff = pd.DataFrame()
else:
    lr_sample = fastccc_long.groupby(['sample_id','group','feature_id','cellpair','sender','receiver','sender_major','receiver_major','lr_pair'], as_index=False).agg(
        lr_strength=('strength','sum'),
        lr_min_pval=('pval','min'),
        lr_percent_ok=('percent_ok','max')
    )
    lr_sample = lr_sample.merge(interaction_map[['lr_pair','ligand_gene','receptor_gene','lr_label']], on='lr_pair', how='left')
    lr_sample['ligand_gene'] = lr_sample['ligand_gene'].fillna(lr_sample['lr_pair'].astype(str))
    lr_sample['receptor_gene'] = lr_sample['receptor_gene'].fillna('unknown')
    lr_sample['lr_label'] = lr_sample['lr_label'].fillna(lr_sample['ligand_gene'].astype(str) + ' -> ' + lr_sample['receptor_gene'].astype(str))
    lr_sample.to_csv(TABLE_DIR / 'lr_sample_summary.tsv', sep='\t', index=False)

    lr_diff_list = []
    for g1, g2 in GROUP_CONTRASTS:
        lr_diff_list.append(differential_by_group(lr_sample, 'feature_id', 'lr_strength', g1, g2))
    lr_diff = pd.concat([x for x in lr_diff_list if x is not None and not x.empty], ignore_index=True) if lr_diff_list else pd.DataFrame()
    if not lr_diff.empty:
        lr_info = lr_sample[['feature_id','cellpair','sender','receiver','sender_major','receiver_major','lr_pair','ligand_gene','receptor_gene','lr_label']].drop_duplicates('feature_id')
        lr_diff = lr_diff.merge(lr_info, on='feature_id', how='left')
        lr_diff.to_csv(TABLE_DIR / 'lr_driver_group_contrasts.tsv', sep='\t', index=False)

print('lr_sample shape:', lr_sample.shape)
print('lr_diff shape:', lr_diff.shape)
display(lr_diff.sort_values(['fdr','pval']).head(50) if not lr_diff.empty else pd.DataFrame({'message':['No LR contrast results.']}))

if not lr_diff.empty:
    plot_lr = lr_diff.sort_values('pval').head(50).copy()
    plot_lr['display_label'] = plot_lr['sender'].astype(str) + ' -> ' + plot_lr['receiver'].astype(str) + ' | ' + plot_lr['lr_label'].astype(str)
    plot_lr['neglog10_p'] = -np.log10(plot_lr['pval'].clip(lower=1e-300))
    label_order = plot_lr.groupby('display_label')['neglog10_p'].max().sort_values().index.tolist()
    plot_lr['display_label'] = pd.Categorical(plot_lr['display_label'], categories=label_order, ordered=True)
    contrast_palette = categorical_palette(plot_lr['contrast'].dropna().unique())

    fig_h = max(4.2, min(10.5, 1.0 + 0.19 * len(label_order)))
    fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))
    sns.scatterplot(
        data=plot_lr,
        x='log2fc_group1_vs_group2',
        y='display_label',
        hue='contrast',
        size='neglog10_p',
        sizes=(22, 170),
        palette=contrast_palette,
        edgecolor='white',
        linewidth=0.65,
        alpha=0.95,
        ax=ax,
        legend='brief',
    )
    ax.axvline(0, color='#202020', lw=1.05, zorder=0)
    ax.set_title('Top ligand-receptor drivers changing between diagnosis groups', loc='left')
    ax.set_xlabel('log2 fold change of FastCCC LR strength')
    ax.set_ylabel('Sender -> receiver | ligand -> receptor')
    wrap_axis_labels(ax, 'y', width=58)
    clean_axes(ax, grid=True, grid_axis='x')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, borderaxespad=0.4)
    fig.subplots_adjust(right=0.75)
    save_pub_figure(fig, FIG_DIR / 'lr_driver_group_contrasts_top50')
    plt.show()


In [ ]:
# ============================================================
# 12. Paper-focused view: astrocytes, CA neurons, neurogenic cells
# ============================================================
PAPER_FOCUS_MAJOR = ['Astrocyte', 'CA_neuron', 'Neurogenic']

if not pair_diff.empty:
    focus_pair_diff = pair_diff[
        pair_diff['sender_major'].isin(PAPER_FOCUS_MAJOR) & pair_diff['receiver_major'].isin(PAPER_FOCUS_MAJOR)
    ].copy()
    focus_pair_diff.to_csv(TABLE_DIR / 'paper_focus_communication_pair_contrasts.tsv', sep='\t', index=False)
else:
    focus_pair_diff = pd.DataFrame()

if not lr_diff.empty:
    focus_lr_diff = lr_diff[
        lr_diff['sender_major'].isin(PAPER_FOCUS_MAJOR) & lr_diff['receiver_major'].isin(PAPER_FOCUS_MAJOR)
    ].copy()
    focus_lr_diff.to_csv(TABLE_DIR / 'paper_focus_lr_driver_contrasts.tsv', sep='\t', index=False)
else:
    focus_lr_diff = pd.DataFrame()

print('Paper focus communication pairs:', focus_pair_diff.shape)
print('Paper focus LR drivers:', focus_lr_diff.shape)
display(focus_pair_diff.sort_values(['fdr','pval']).head(30) if not focus_pair_diff.empty else pd.DataFrame({'message':['No paper-focus pair results.']}))
display(focus_lr_diff.sort_values(['fdr','pval']).head(50) if not focus_lr_diff.empty else pd.DataFrame({'message':['No paper-focus LR results.']}))

if not focus_lr_diff.empty:
    for contrast in focus_lr_diff['contrast'].dropna().unique():
        sub = focus_lr_diff[focus_lr_diff['contrast'].eq(contrast)].sort_values('pval').head(28).copy()
        sub['display_label'] = sub['sender'].astype(str) + ' -> ' + sub['receiver'].astype(str) + ' | ' + sub['lr_label'].astype(str)
        sub['neglog10_p'] = -np.log10(sub['pval'].clip(lower=1e-300))
        sub = sub.iloc[::-1].reset_index(drop=True)
        y = np.arange(len(sub))
        colors = np.where(sub['log2fc_group1_vs_group2'] >= 0, '#D55E00', '#0072B2')

        fig_h = max(3.6, min(8.5, 1.0 + 0.24 * len(sub)))
        fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))
        ax.hlines(y=y, xmin=0, xmax=sub['log2fc_group1_vs_group2'], color=colors, lw=1.1, alpha=0.85)
        ax.scatter(
            sub['log2fc_group1_vs_group2'], y,
            s=scaled_sizes(sub['neglog10_p'], min_size=25, max_size=210),
            c=colors,
            edgecolor='white',
            linewidth=0.655,
            zorder=3,
        )
        ax.axvline(0, color='#202020', lw=1.05, zorder=0)
        ax.set_yticks(y)
        ax.set_yticklabels(sub['display_label'])
        wrap_axis_labels(ax, 'y', width=58)
        ax.set_title(f'Paper-focused LR drivers: {contrast}', loc='left')
        ax.set_xlabel('log2 fold change of FastCCC LR strength')
        ax.set_ylabel('Sender -> receiver | ligand -> receptor')
        clean_axes(ax, grid=True, grid_axis='x')
        handles = [
            Line2D([0], [0], color='#D55E00', marker='o', lw=1.1, label='Higher in group 1'),
            Line2D([0], [0], color='#0072B2', marker='o', lw=1.1, label='Higher in group 2'),
        ]
        ax.legend(handles=handles, loc='lower right', frameon=False)
        save_pub_figure(fig, FIG_DIR / f'paper_focus_lr_driver_lollipop_{safe_filename(contrast)}')
        plt.show()


In [ ]:
# ============================================================
# 13. Latent-time window FastCCC for neurogenic trajectory
# ============================================================
N_LATENT_WINDOWS = 8
LATENT_MIN_CELLS_PER_LABEL = 80
LATENT_MAX_CELLS_PER_LABEL = 700
LATENT_KEEP_MAJOR = ['Neurogenic', 'Astrocyte', 'CA_neuron', 'Microglia', 'OPC', 'Oligodendrocyte', 'Vascular']

latent_obs = adata.obs.copy()
latent_obs['latent_ccc_celltype'] = latent_obs['major_celltype'].astype(str)
latent_obs['latent_time_numeric'] = pd.to_numeric(latent_obs[LATENT_TIME_KEY], errors='coerce') if LATENT_TIME_KEY in latent_obs.columns else np.nan

neuro_mask = latent_obs['major_celltype'].eq('Neurogenic') & latent_obs['latent_time_numeric'].notna()
neuro_time = latent_obs.loc[neuro_mask, 'latent_time_numeric']
if neuro_time.shape[0] > 0:
    ranked_time = neuro_time.rank(method='first')
    n_bins = min(N_LATENT_WINDOWS, int(neuro_time.nunique()))
    bin_id = pd.qcut(ranked_time, q=n_bins, labels=False, duplicates='drop')
    bin_id = pd.Series(bin_id, index=neuro_time.index).astype(int)
    latent_obs.loc[neuro_mask, 'latent_bin'] = bin_id
    latent_obs.loc[neuro_mask, 'latent_ccc_celltype'] = bin_id.map(lambda x: f'Neurogenic_t{x+1:02d}')
else:
    latent_obs['latent_bin'] = np.nan

adata.obs['latent_ccc_celltype'] = latent_obs['latent_ccc_celltype'].astype(str)
adata.obs['latent_time_numeric'] = latent_obs['latent_time_numeric']

latent_counts = latent_obs['latent_ccc_celltype'].value_counts().rename_axis('latent_ccc_celltype').reset_index(name='n_cells')
latent_counts.to_csv(TABLE_DIR / 'latent_ccc_celltype_counts.tsv', sep='\t', index=False)
display(latent_counts)

keep_labels = latent_counts.loc[latent_counts['n_cells'] >= LATENT_MIN_CELLS_PER_LABEL, 'latent_ccc_celltype'].tolist()
keep_labels = [x for x in keep_labels if x.startswith('Neurogenic_t') or infer_hippo_major(x, None) in LATENT_KEEP_MAJOR]
print('Labels kept for latent-time FastCCC:', keep_labels)

latent_input_path = TMP / 'latent_time_window_fastccc_input.h5ad'
latent_save_dir = LATENT_DIR / 'fastccc_latent_time_windows'

if RUN_LATENT_WINDOW_FASTCCC:
    latent_sub = adata[adata.obs['latent_ccc_celltype'].isin(keep_labels)].copy()
    latent_sub = downsample_by_celltype(latent_sub, 'latent_ccc_celltype', LATENT_MAX_CELLS_PER_LABEL, random_state=2026)
    latent_sub = normalize_for_fastccc(latent_sub)
    latent_sub.write_h5ad(latent_input_path)
    latent_strength, latent_pvals, latent_percents, latent_status = run_fastccc_on_h5ad(
        latent_input_path,
        latent_save_dir,
        meta_key='latent_ccc_celltype',
        rerun=RERUN_LATENT_EXISTING,
    )
    print('Latent-time FastCCC status:', latent_status)
    print('Latent-time FastCCC matrix:', latent_strength.shape)
else:
    print('RUN_LATENT_WINDOW_FASTCCC is False. This cell only prepared labels.')


In [ ]:
# ============================================================
# 14. Visualize neurogenic latent-time communication
# ============================================================
def extract_latent_fastccc_long(save_dir, pval_cutoff=PVAL_SIG_CUTOFF):
    save_dir = Path(save_dir)
    strength = pd.read_csv(save_dir / 'interactions_strength.tsv', sep='\t', index_col=0)
    pvals = pd.read_csv(save_dir / 'pvals.tsv', sep='\t', index_col=0)
    percents = pd.read_csv(save_dir / 'percents_analysis.tsv', sep='\t', index_col=0)
    row_idx, col_idx = np.where(pvals.values < pval_cutoff)
    df = pd.DataFrame({
        'cellpair': strength.index.to_numpy()[row_idx],
        'lr_pair': strength.columns.to_numpy()[col_idx],
        'strength': strength.values[row_idx, col_idx],
        'pval': pvals.values[row_idx, col_idx],
        'percent_ok': percents.values[row_idx, col_idx],
    })
    if df.empty:
        return df
    df[['sender','receiver']] = df['cellpair'].apply(lambda x: pd.Series(parse_cellpair(x)))
    df = df.merge(interaction_map[['lr_pair','ligand_gene','receptor_gene','lr_label']], on='lr_pair', how='left')
    df['ligand_gene'] = df['ligand_gene'].fillna(df['lr_pair'].astype(str))
    df['receptor_gene'] = df['receptor_gene'].fillna('unknown')
    df['lr_label'] = df['lr_label'].fillna(df['ligand_gene'].astype(str) + ' -> ' + df['receptor_gene'].astype(str))
    return df


def latent_window_number(label):
    m = re.search(r'Neurogenic_t(\d+)', str(label))
    if m:
        return int(m.group(1))
    return np.nan


if (latent_save_dir / 'interactions_strength.tsv').exists():
    latent_long = extract_latent_fastccc_long(latent_save_dir)
    if not latent_long.empty:
        latent_long['sender_window'] = latent_long['sender'].map(latent_window_number)
        latent_long['receiver_window'] = latent_long['receiver'].map(latent_window_number)
        latent_long['neurogenic_role'] = np.select(
            [latent_long['sender_window'].notna(), latent_long['receiver_window'].notna()],
            ['neurogenic_sender', 'neurogenic_receiver'],
            default='other'
        )
        latent_long['latent_window'] = latent_long['sender_window'].fillna(latent_long['receiver_window'])
        latent_long['partner_celltype'] = np.where(latent_long['sender_window'].notna(), latent_long['receiver'], latent_long['sender'])
        latent_focus = latent_long[latent_long['latent_window'].notna()].copy()
        latent_focus.to_csv(TABLE_DIR / 'latent_time_neurogenic_fastccc_long.tsv', sep='\t', index=False)
        print('latent_focus shape:', latent_focus.shape)
        display(latent_focus.sort_values(['pval','strength'], ascending=[True, False]).head(40))

        latent_pair_summary = latent_focus.groupby(['neurogenic_role','latent_window','partner_celltype'], as_index=False).agg(
            n_sig_lr=('lr_pair','nunique'),
            sum_strength=('strength','sum'),
            min_pval=('pval','min')
        )
        latent_pair_summary.to_csv(TABLE_DIR / 'latent_time_partner_summary.tsv', sep='\t', index=False)

        top_partners = latent_pair_summary.groupby('partner_celltype', as_index=False)['sum_strength'].sum().sort_values('sum_strength', ascending=False).head(18)['partner_celltype'].tolist()
        plot_latent = latent_pair_summary[latent_pair_summary['partner_celltype'].isin(top_partners)].copy()
        plot_latent['neglog10_p'] = -np.log10(plot_latent['min_pval'].clip(lower=1e-300))
        partner_order = plot_latent.groupby('partner_celltype')['sum_strength'].sum().sort_values().index.tolist()
        y_map = {p: i for i, p in enumerate(partner_order)}
        plot_latent['y'] = plot_latent['partner_celltype'].map(y_map)
        role_markers = {'neurogenic_sender': 'o', 'neurogenic_receiver': '^'}

        fig_h = max(3.8, min(7.5, 1.0 + 0.25 * len(partner_order)))
        fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))
        norm = mpl.colors.Normalize(vmin=0, vmax=max(float(plot_latent['sum_strength'].max()), 1e-9))
        sca = None
        for role, marker in role_markers.items():
            sub = plot_latent[plot_latent['neurogenic_role'].eq(role)]
            if sub.empty:
                continue
            sca = ax.scatter(
                sub['latent_window'], sub['y'],
                s=scaled_sizes(sub['n_sig_lr'], min_size=24, max_size=220),
                c=sub['sum_strength'], cmap='viridis', norm=norm,
                marker=marker, edgecolors='white', linewidths=1.4,
                alpha=0.95, label=role.replace('_', ' '),
            )
        ax.set_yticks(range(len(partner_order)))
        ax.set_yticklabels(partner_order)
        wrap_axis_labels(ax, 'y', width=34)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_title('Neurogenic latent-time communication with partner cell types', loc='left')
        ax.set_xlabel('Neurogenic latent-time window')
        ax.set_ylabel('Partner cell type')
        clean_axes(ax, grid=True, grid_axis='x')
        if sca is not None:
            cbar = fig.colorbar(sca, ax=ax, pad=0.012, fraction=0.035)
            cbar.set_label('Summed FastCCC strength')
        role_legend = ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, title='Neurogenic role')
        ax.add_artist(role_legend)
        add_size_legend(ax, plot_latent['n_sig_lr'], 'No. significant LRIs',
                        min_size=24, max_size=220, bbox_to_anchor=(1.18, 0.55))
        fig.subplots_adjust(right=0.80)
        save_pub_figure(fig, FIG_DIR / 'latent_time_neurogenic_partner_dotplot')
        plt.show()

        top_lrs = latent_focus.groupby('lr_label', as_index=False).agg(n_windows=('latent_window','nunique'), total_strength=('strength','sum'), best_pval=('pval','min')).sort_values(['n_windows','total_strength'], ascending=False).head(20)['lr_label'].tolist()
        lr_plot = latent_focus[latent_focus['lr_label'].isin(top_lrs)].groupby(['latent_window','lr_label'], as_index=False).agg(sum_strength=('strength','sum'), min_pval=('pval','min'))
        lr_plot['neglog10_p'] = -np.log10(lr_plot['min_pval'].clip(lower=1e-300))
        lr_order = lr_plot.groupby('lr_label')['sum_strength'].sum().sort_values().index.tolist()
        lr_plot['y'] = lr_plot['lr_label'].map({lr: i for i, lr in enumerate(lr_order)})

        fig_h = max(3.8, min(8.0, 1.0 + 0.25 * len(lr_order)))
        fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))
        norm = mpl.colors.Normalize(vmin=0, vmax=max(float(lr_plot['sum_strength'].max()), 1e-9))
        sca = ax.scatter(
            lr_plot['latent_window'], lr_plot['y'],
            s=scaled_sizes(lr_plot['neglog10_p'], min_size=45, max_size=260),
            c=lr_plot['sum_strength'], cmap='magma', norm=norm,
            edgecolors='white', linewidths=0.65, alpha=0.95,
        )
        ax.set_yticks(range(len(lr_order)))
        ax.set_yticklabels(lr_order)
        wrap_axis_labels(ax, 'y', width=42)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_title('Top ligand-receptor interactions along neurogenic latent time', loc='left')
        ax.set_xlabel('Neurogenic latent-time window')
        ax.set_ylabel('Ligand -> receptor')
        clean_axes(ax, grid=True, grid_axis='x')
        cbar = fig.colorbar(sca, ax=ax, pad=0.012, fraction=0.035)
        cbar.set_label('Summed FastCCC strength')
        add_size_legend(ax, lr_plot['neglog10_p'], '-log10 p', min_size=45, max_size=260,
                        bbox_to_anchor=(1.16, 1.0))
        fig.subplots_adjust(right=0.82)
        save_pub_figure(fig, FIG_DIR / 'latent_time_top_lr_dotplot')
        plt.show()
    else:
        latent_focus = pd.DataFrame()
        print('No significant latent-time FastCCC interactions under current cutoff.')
else:
    latent_focus = pd.DataFrame()
    print('Latent-time FastCCC output not found. Run cell 13 first.')


In [ ]:
# ============================================================
# 15. Functional enrichment for changing ligand-receptor genes
# ============================================================
def extract_gene_symbols_from_lr_name(name):
    s = str(name).replace('_HUMAN', '')
    if '_by' in s:
        s = s.split('_by', 1)[1]
    s = s.replace('and', '_')
    tokens = re.findall(r'[A-Za-z][A-Za-z0-9]{1,12}', s)
    cleaned = []
    stop = {'HUMAN', 'RECEPTOR', 'COMPLEX', 'UNKNOWN', 'BY'}
    for t in tokens:
        u = t.upper()
        if u in stop:
            continue
        if len(u) < 2:
            continue
        cleaned.append(u)
    return sorted(set(cleaned))


def genes_from_lr_table(df, top_n=150):
    if df is None or df.empty:
        return []
    sub = df.sort_values('pval').head(top_n).copy()
    genes = []
    for col in ['ligand_gene', 'receptor_gene']:
        for x in sub[col].dropna().astype(str):
            genes.extend(extract_gene_symbols_from_lr_name(x))
    return sorted(set(genes))


def run_enrichr_for_genes(gene_list, label, libraries=None):
    if libraries is None:
        libraries = ['GO_Biological_Process_2023', 'KEGG_2021_Human', 'Reactome_2022']
    out_gene_file = ENRICH_DIR / f'{label}_genes.txt'
    out_gene_file.write_text('\n'.join(gene_list), encoding='utf-8')
    if len(gene_list) < 5:
        print(f'{label}: too few genes for enrichment:', len(gene_list))
        return pd.DataFrame()
    try:
        import gseapy as gp
    except Exception as e:
        print('gseapy is not available. Gene list saved:', out_gene_file)
        print('Import error:', repr(e))
        return pd.DataFrame()

    all_res = []
    for lib in libraries:
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=lib,
                organism='human',
                outdir=None,
                cutoff=1.0,
            )
            res = enr.results.copy()
            res['library'] = lib
            res['label'] = label
            all_res.append(res)
        except Exception as e:
            print(f'Enrichr failed for {label} / {lib}:', repr(e))
    if not all_res:
        return pd.DataFrame()
    out = pd.concat(all_res, ignore_index=True)
    out.to_csv(ENRICH_DIR / f'{label}_enrichr.tsv', sep='\t', index=False)
    return out

enrich_frames = []
if 'focus_lr_diff' in globals() and not focus_lr_diff.empty:
    for contrast in focus_lr_diff['contrast'].dropna().unique():
        for direction in ['higher_group_1', 'higher_group_2']:
            if direction == 'higher_group_1':
                sub = focus_lr_diff[(focus_lr_diff['contrast'].eq(contrast)) & (focus_lr_diff['log2fc_group1_vs_group2'] > 0)].copy()
            else:
                sub = focus_lr_diff[(focus_lr_diff['contrast'].eq(contrast)) & (focus_lr_diff['log2fc_group1_vs_group2'] < 0)].copy()
            genes = genes_from_lr_table(sub, top_n=150)
            label = f'paper_focus_{contrast}_{direction}'
            print(label, 'n_genes =', len(genes))
            res = run_enrichr_for_genes(genes, label)
            if not res.empty:
                enrich_frames.append(res)

if 'latent_focus' in globals() and not latent_focus.empty:
    genes = genes_from_lr_table(latent_focus.sort_values(['pval','strength'], ascending=[True, False]), top_n=200)
    print('latent_time_neurogenic n_genes =', len(genes))
    res = run_enrichr_for_genes(genes, 'latent_time_neurogenic')
    if not res.empty:
        enrich_frames.append(res)

enrichment_results = pd.concat(enrich_frames, ignore_index=True) if enrich_frames else pd.DataFrame()
if not enrichment_results.empty:
    enrichment_results.to_csv(ENRICH_DIR / 'all_enrichment_results.tsv', sep='\t', index=False)

display(enrichment_results.head(30) if not enrichment_results.empty else pd.DataFrame({'message':['No enrichment result available. Gene lists were still saved if genes existed.']}))


In [ ]:
# ============================================================
# 16. Enrichment dotplot
# ============================================================
if 'enrichment_results' in globals() and not enrichment_results.empty:
    enr = enrichment_results.copy()
    p_col = 'Adjusted P-value' if 'Adjusted P-value' in enr.columns else ('P-value' if 'P-value' in enr.columns else None)
    term_col = 'Term' if 'Term' in enr.columns else enr.columns[0]
    if p_col is not None:
        enr['neglog10_adj_p'] = -np.log10(pd.to_numeric(enr[p_col], errors='coerce').clip(lower=1e-300))
        enr['term_short'] = enr[term_col].astype(str).str.replace(r' \(GO.*\)$', '', regex=True).str.slice(0, 80)
        plot_enr = enr.sort_values(p_col).groupby('label', as_index=False).head(7).copy()
        term_order = plot_enr.groupby('term_short')['neglog10_adj_p'].max().sort_values().index.tolist()
        plot_enr['term_short'] = pd.Categorical(plot_enr['term_short'], categories=term_order, ordered=True)
        label_palette = categorical_palette(plot_enr['label'].dropna().unique())

        fig_h = max(3.8, min(9.0, 1.0 + 0.22 * len(term_order)))
        fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))
        sns.scatterplot(
            data=plot_enr,
            x='neglog10_adj_p',
            y='term_short',
            hue='label',
            size='neglog10_adj_p',
            sizes=(24, 190),
            palette=label_palette,
            edgecolor='white',
            linewidth=0.655,
            alpha=0.95,
            ax=ax,
            legend='brief',
        )
        ax.set_title('Functional enrichment of changing LR genes', loc='left')
        ax.set_xlabel('-log10 adjusted p-value')
        ax.set_ylabel('Pathway or GO term')
        wrap_axis_labels(ax, 'y', width=52)
        clean_axes(ax, grid=True, grid_axis='x')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, borderaxespad=0.4)
        fig.subplots_adjust(right=0.76)
        save_pub_figure(fig, FIG_DIR / 'lr_gene_enrichment_dotplot')
        plt.show()
    else:
        print('Could not identify p-value column in enrichment table.')
else:
    print('No enrichment results to plot.')


In [ ]:
# ============================================================
# 17. Ligand and receptor expression validation by diagnosis group
# ============================================================
def find_gene_in_adata(symbol):
    candidates = extract_gene_symbols_from_lr_name(symbol)
    var_upper = pd.Index(adata.var_names.astype(str).str.upper())
    for g in candidates:
        hits = np.where(var_upper == g.upper())[0]
        if len(hits) > 0:
            return adata.var_names[int(hits[0])]
    return None


def mean_gene_expr(mask, gene):
    if gene is None or int(np.sum(mask)) == 0:
        return np.nan
    x = adata[mask, gene].X
    try:
        return float(x.mean())
    except Exception:
        return float(np.asarray(x).mean())


if 'focus_lr_diff' in globals() and not focus_lr_diff.empty:
    top_validate = focus_lr_diff.sort_values('pval').head(12).copy()
    expr_rows = []
    for _, r in top_validate.iterrows():
        ligand_var = find_gene_in_adata(r['ligand_gene'])
        receptor_var = find_gene_in_adata(r['receptor_gene'])
        obs_group = normalize_group_series(adata.obs['diagnosis_group'].astype(str)).values
        for group in [g for g in GROUP_ORDER if g in set(obs_group)]:
            sender_mask = pd.Series(obs_group, index=adata.obs.index).eq(group) & adata.obs['ccc_celltype'].astype(str).eq(str(r['sender']))
            receiver_mask = pd.Series(obs_group, index=adata.obs.index).eq(group) & adata.obs['ccc_celltype'].astype(str).eq(str(r['receiver']))
            expr_rows.append({
                'feature_id': r['feature_id'],
                'contrast': r['contrast'],
                'group': group,
                'sender': r['sender'],
                'receiver': r['receiver'],
                'lr_label': r['lr_label'],
                'ligand_gene': r['ligand_gene'],
                'receptor_gene': r['receptor_gene'],
                'ligand_var': ligand_var,
                'receptor_var': receptor_var,
                'ligand_mean_in_sender': mean_gene_expr(sender_mask.values, ligand_var),
                'receptor_mean_in_receiver': mean_gene_expr(receiver_mask.values, receptor_var),
                'n_sender_cells': int(sender_mask.sum()),
                'n_receiver_cells': int(receiver_mask.sum()),
            })
    expression_validation = pd.DataFrame(expr_rows)
    expression_validation.to_csv(TABLE_DIR / 'top_lr_expression_validation_by_group.tsv', sep='\t', index=False)
    display(expression_validation.head(40))

    plot_expr = expression_validation.melt(
        id_vars=['feature_id','contrast','group','sender','receiver','lr_label'],
        value_vars=['ligand_mean_in_sender','receptor_mean_in_receiver'],
        var_name='molecule_role',
        value_name='mean_expression'
    )
    plot_expr['molecule_role'] = plot_expr['molecule_role'].map({
        'ligand_mean_in_sender': 'Ligand in sender',
        'receptor_mean_in_receiver': 'Receptor in receiver',
    })
    plot_expr['label'] = plot_expr['sender'].astype(str) + ' -> ' + plot_expr['receiver'].astype(str) + ' | ' + plot_expr['lr_label'].astype(str)
    plot_expr['group'] = pd.Categorical(plot_expr['group'], categories=ordered_groups(plot_expr['group']), ordered=True)
    for contrast in plot_expr['contrast'].dropna().unique():
        sub = plot_expr[plot_expr['contrast'].eq(contrast)].copy()
        keep_labels = sub.groupby('label')['mean_expression'].max().dropna().nlargest(8).index.tolist()
        if not keep_labels:
            print(f'Skip {contrast}: no expression signal to plot.')
            continue
        sub = sub[sub['label'].isin(keep_labels)].copy()
        label_palette = categorical_palette(keep_labels)

        fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, max(3.4, 0.25 * len(keep_labels) + 2.2)))
        sns.lineplot(
            data=sub,
            x='group',
            y='mean_expression',
            hue='label',
            style='molecule_role',
            markers=True,
            dashes=True,
            palette=label_palette,
            linewidth=1.2,
            markersize=8,
            ax=ax,
        )
        ax.set_title(f'Expression validation for top LR drivers: {contrast}', loc='left')
        ax.set_xlabel('Diagnosis group')
        ax.set_ylabel('Mean expression in sender or receiver cells')
        clean_axes(ax, grid=True, grid_axis='y')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=12, borderaxespad=0.4)
        fig.subplots_adjust(right=0.68)
        save_pub_figure(fig, FIG_DIR / f'top_lr_expression_validation_{safe_filename(contrast)}')
        plt.show()
else:
    expression_validation = pd.DataFrame()
    print('No focus LR results for expression validation.')


In [ ]:
# ============================================================
# 18. Load per-sample full matrices + group-level aggregation
# ============================================================
print('Loading per-sample FastCCC full matrices...')
all_chunks = []
for _, row in run_summary[run_summary['status'].isin(['ok', 'exists_skip'])].iterrows():
    sid = str(row['sample_id'])
    grp = str(row['group'])
    sample_dir = RESULTS / sid
    pv = pd.read_csv(sample_dir / 'pvals.tsv', sep='\t', index_col=0)
    st = pd.read_csv(sample_dir / 'interactions_strength.tsv', sep='\t', index_col=0)
    common_idx = pv.index.intersection(st.index)
    common_col = pv.columns.intersection(st.columns)
    pv = pv.loc[common_idx, common_col]
    st = st.loc[common_idx, common_col]
    pv_long = pv.stack().reset_index()
    pv_long.columns = ['cellpair', 'lr_pair', 'pval']
    st_long = st.stack().reset_index()
    st_long.columns = ['cellpair', 'lr_pair', 'strength']
    chunk = pv_long.merge(st_long, on=['cellpair', 'lr_pair'])
    chunk['sample_id'] = sid
    chunk['group'] = grp
    all_chunks.append(chunk)
    print(f'  {sid} ({grp}): {len(chunk):,} entries')

all_full = pd.concat(all_chunks, ignore_index=True)
del all_chunks; gc.collect()
all_full['is_sig'] = (all_full['pval'] < PVAL_SIG_CUTOFF).astype(int)
all_full[['sender', 'receiver']] = all_full['cellpair'].apply(lambda x: pd.Series(parse_cellpair(x)))
all_full = all_full.merge(
    interaction_map[['lr_pair','ligand_gene','receptor_gene','lr_label']].drop_duplicates('lr_pair'),
    on='lr_pair', how='left'
)
all_full['lr_label'] = all_full['lr_label'].fillna(all_full['lr_pair'].astype(str))

# Group-level aggregation
group_lr = all_full.groupby(
    ['group','cellpair','lr_pair','sender','receiver','lr_label'], as_index=False
).agg(
    n_samples=('sample_id','nunique'),
    mean_pval=('pval','mean'),
    mean_strength=('strength','mean'),
    n_sig=('is_sig','sum'),
)
group_lr['frac_sig'] = group_lr['n_sig'] / group_lr['n_samples']
group_lr['neglog10_p'] = -np.log10(group_lr['mean_pval'].clip(lower=1e-300))

print(f'\nall_full: {all_full.shape[0]:,} rows')
print(f'group_lr: {group_lr.shape[0]:,} rows')
print(f'Groups: {sorted(all_full["group"].unique())}')


In [ ]:
# ============================================================
# 19. Per-group significant LRI counts + Venn diagram
# ============================================================
sig_full = all_full[all_full['is_sig'] == 1].copy()
sig_full['group'] = normalize_group_series(sig_full['group']).values
sig_full['feature_id'] = sig_full['cellpair'] + ' || ' + sig_full['lr_pair']

groups_present = ordered_groups(sig_full['group'])
group_sig_count = sig_full.groupby('group').agg(
    n_sig_lri=('feature_id', 'nunique'),
    n_sig_cellpairs=('cellpair', 'nunique'),
    n_sig_lr_pairs=('lr_pair', 'nunique'),
).reindex(groups_present).dropna(how='all')

print(group_sig_count)

if group_sig_count.empty or group_sig_count['n_sig_lri'].fillna(0).sum() == 0:
    print('No significant LRI counts to plot.')
else:
    fig, ax = plt.subplots(figsize=(FIG_SINGLE_WIDTH, 2.65))
    pal = group_palette(group_sig_count.index)
    bars = ax.bar(group_sig_count.index, group_sig_count['n_sig_lri'],
                  color=[pal[g] for g in group_sig_count.index],
                  edgecolor='white', linewidth=0.85, width=0.68)
    ax.set_ylabel('Significant LRIs')
    ax.set_xlabel('Diagnosis group')
    ax.set_title('Significant ligand-receptor interactions per group', loc='left')
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    for bar, v in zip(bars, group_sig_count['n_sig_lri']):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{int(v):,}',
                ha='center', va='bottom', fontsize=12, color='#303030')
    clean_axes(ax, grid=True, grid_axis='y')
    save_pub_figure(fig, FIG_DIR / 'fig4b_group_sig_lri_barplot')
    plt.show()

CONTRASTS_FIG4 = [('YA','HA'), ('HA','AD'), ('HA','SA'), ('AD','SA')]
lri_sets = {g: set(sig_full.loc[sig_full['group'].eq(g), 'feature_id'].unique()) for g in GROUP_ORDER}
valid_contrasts = [(g1, g2) for g1, g2 in CONTRASTS_FIG4 if len(lri_sets.get(g1, set())) > 0 and len(lri_sets.get(g2, set())) > 0]

try:
    from matplotlib_venn import venn2
    if not valid_contrasts:
        print('No non-empty group pairs for Venn plots.')
    else:
        ncols = 2 if len(valid_contrasts) > 1 else 1
        nrows = int(np.ceil(len(valid_contrasts) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(max(FIG_DOUBLE_WIDTH, 13.5), 4.0 * nrows))
        axes = np.atleast_1d(axes).ravel()
        for ax, (g1, g2) in zip(axes, valid_contrasts):
            venn2(
                [lri_sets[g1], lri_sets[g2]],
                set_labels=(g1, g2),
                set_colors=(GROUP_COLORS.get(g1, '#999999'), GROUP_COLORS.get(g2, '#BBBBBB')),
                alpha=0.65,
                ax=ax,
            )
            ax.set_title(f'{g1} vs {g2}', fontsize=12, weight='bold')
        for ax in axes[len(valid_contrasts):]:
            ax.axis('off')
        fig.suptitle('Shared and unique significant LRIs between groups', fontsize=12, fontweight='bold', y=0.99)
        save_pub_figure(fig, FIG_DIR / 'fig4c_venn_lri')
        plt.show()
except ImportError:
    print('matplotlib_venn not installed. Showing overlap table instead.')
    rows = []
    for g1, g2 in valid_contrasts:
        s1, s2 = lri_sets.get(g1, set()), lri_sets.get(g2, set())
        rows.append({'contrast': f'{g1} vs {g2}', f'only_{g1}': len(s1 - s2), 'shared': len(s1 & s2), f'only_{g2}': len(s2 - s1)})
    display(pd.DataFrame(rows) if rows else pd.DataFrame({'message': ['No non-empty group pairs.']}))


In [ ]:
# ============================================================
# 20a. Fig.4h — 双组比较 LRI 气泡图（第一段代码配色）
# ============================================================
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.transforms import blended_transform_factory
from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox

CONTRASTS_FIG4 = [('YA','HA'), ('HA','AD'), ('HA','SA'), ('AD','SA')]
GLOBAL_VMAX = 3.0

# ── 配色（对齐第一段代码）──
COLOR_G1   = np.array(mcolors.to_rgb("#2F80ED"))   # bright blue
COLOR_G2   = np.array(mcolors.to_rgb("#EB5757"))   # coral red
COLOR_BOTH = np.array(mcolors.to_rgb("#9B51E0"))   # violet
COLOR_WEAK = np.array(mcolors.to_rgb("#F3F3F3"))   # neutral low-signal gray
LIGAND_COLOR   = "#2F80ED"
RECEPTOR_COLOR = "#EB5757"

def _norm_s(v, cap):
    return (np.clip(float(v), 0, cap) / cap) ** 0.55

def _norm_d(d, cap):
    return np.clip(float(d), -cap, cap) / cap

def _dot_style(nlp1, nlp2, vmax=GLOBAL_VMAX):
    """第一段代码的 group_contrast_color + group_contrast_size 逻辑"""
    g1, g2 = float(nlp1), float(nlp2)
    strength = _norm_s(g1 + g2, vmax * 2)
    diff = _norm_d(g1 - g2, vmax)

    # 大小
    size = 12 + 220 * strength

    # 颜色
    if strength < 0.12:
        rgb = COLOR_WEAK
    else:
        if diff > 0.15:
            dom = min(abs(diff), 1.0)
            rgb = (1 - dom) * COLOR_BOTH + dom * COLOR_G1
        elif diff < -0.15:
            dom = min(abs(diff), 1.0)
            rgb = (1 - dom) * COLOR_BOTH + dom * COLOR_G2
        else:
            rgb = COLOR_BOTH
        rgb = COLOR_WEAK * (1 - strength) + rgb * strength

    rgb = np.clip(rgb, 0, 1)
    return (*rgb, 1.0), size

def _make_legend_img(vmax=GLOBAL_VMAX, N=50):
    img = np.zeros((N, N, 4))
    vals = np.linspace(0, vmax, N)
    for ix in range(N):
        for iy in range(N):
            rgba, _ = _dot_style(vals[iy], vals[ix], vmax)
            img[N-1-iy, ix] = rgba
    return img

def fig4h_plot(group_lr, g1, g2, top_n=30, min_frac=0.05, save_path=None):
    cols = ['cellpair','lr_pair','lr_label','sender','receiver',
            'mean_pval','frac_sig','neglog10_p']
    d1 = group_lr.loc[group_lr['group']==g1, cols].copy()
    d2 = group_lr.loc[group_lr['group']==g2,
         ['cellpair','lr_pair','mean_pval','frac_sig','neglog10_p']].copy()
    mg = d1.merge(d2, on=['cellpair','lr_pair'],
                  suffixes=(f'_{g1}',f'_{g2}'), how='outer')
    for s in [g1,g2]:
        mg[f'mean_pval_{s}']  = mg[f'mean_pval_{s}'].fillna(1.0)
        mg[f'frac_sig_{s}']   = mg[f'frac_sig_{s}'].fillna(0.0)
        mg[f'neglog10_p_{s}'] = mg[f'neglog10_p_{s}'].fillna(0.0)
    na = mg['sender'].isna()
    if na.any():
        f = mg.loc[na,'cellpair'].apply(lambda c: pd.Series(parse_cellpair(c)))
        mg.loc[na,'sender']=f[0].values; mg.loc[na,'receiver']=f[1].values
    if mg['lr_label'].isna().any():
        lm = group_lr[['lr_pair','lr_label']].drop_duplicates('lr_pair')
        mg = mg.drop(columns='lr_label').merge(lm, on='lr_pair', how='left')
        mg['lr_label'] = mg['lr_label'].fillna(mg['lr_pair'])

    mg['max_nlp'] = mg[[f'neglog10_p_{g1}',f'neglog10_p_{g2}']].max(axis=1)
    mg = mg[(mg[f'frac_sig_{g1}']>=min_frac)|(mg[f'frac_sig_{g2}']>=min_frac)].copy()
    if mg.empty: print(f'No LRI for {g1} vs {g2}'); return None

    top_lr = mg.groupby('lr_label')['max_nlp'].max().nlargest(top_n).index.tolist()
    mg = mg[mg['lr_label'].isin(top_lr)]
    lr_order = mg.groupby('lr_label')['max_nlp'].max().sort_values(ascending=True).index.tolist()
    active_cp = set(mg['cellpair'].unique())
    cp_order = (mg[['cellpair','sender','receiver']].drop_duplicates()
                .sort_values(['sender','receiver'])['cellpair'].tolist())
    cp_order = [c for c in cp_order if c in active_cp]
    n_y, n_x = len(lr_order), len(cp_order)
    if n_x==0 or n_y==0: print(f'Empty for {g1} vs {g2}'); return None

    col_w = max(0.30, min(0.55, 18/n_x))
    row_h = max(0.28, min(0.42, 14/n_y))
    fig, ax = plt.subplots(figsize=(max(10, n_x*col_w+5), max(6, n_y*row_h+3.8)))
    fig.patch.set_facecolor('white'); ax.set_facecolor('white')

    # ── 气泡 ──
    for _, r in mg.iterrows():
        if r['lr_label'] not in lr_order or r['cellpair'] not in cp_order: continue
        xi, yi = cp_order.index(r['cellpair']), lr_order.index(r['lr_label'])
        n1, n2 = r[f'neglog10_p_{g1}'], r[f'neglog10_p_{g2}']
        rgba, sz = _dot_style(n1, n2)
        ax.scatter(xi, yi, s=sz, c=[rgba[:3]], alpha=rgba[3],
                   edgecolors='k' if max(n1,n2)>0.3 else 'none',
                   linewidths=0.6, zorder=3)

    # ── 竖虚线 ──
    senders = [parse_cellpair(c)[0] for c in cp_order]
    for i in range(1, n_x):
        if senders[i] != senders[i-1]:
            ax.axvline(i-0.5, color='#aaa', ls='--', lw=1.05, zorder=1)
    for yi in range(n_y):
        ax.axhline(yi, color='#eee', lw=0.95, zorder=0)

    ax.set_xlim(-0.5, n_x-0.5); ax.set_ylim(-0.5, n_y-0.5)
    ax.set_xticks(range(n_x)); ax.set_xticklabels(['']*n_x)
    ax.set_yticks(range(n_y)); ax.set_yticklabels(['']*n_y)
    ax.tick_params(length=0)
    for sp in ax.spines.values(): sp.set_visible(False)

    # ── Y 轴：ligand–receptor 双色 ──
    for i, lr in enumerate(lr_order):
        pts = [p.strip() for p in lr.split(' -> ')]
        if len(pts)==2:
            ch = [TextArea(pts[0], textprops=dict(color=LIGAND_COLOR,size=10,weight='bold')),
                  TextArea('–',    textprops=dict(color='#555',size=10)),
                  TextArea(pts[1], textprops=dict(color=RECEPTOR_COLOR,size=10,weight='bold'))]
        else:
            ch = [TextArea(lr, textprops=dict(color='k',size=10))]
        ab = AnnotationBbox(HPacker(children=ch,pad=0,sep=1,align='center'),
             xy=(-0.005,i), xycoords=('axes fraction','data'),
             box_alignment=(1.0,0.5), frameon=False)
        ax.add_artist(ab)

    # ── X 轴上排 receiver（红斜）──
    xt = blended_transform_factory(ax.transData, ax.transAxes)
    for i, cp in enumerate(cp_order):
        _, recv = parse_cellpair(cp)
        ax.text(i, -0.015, recv, transform=xt, ha='right', va='top',
                rotation=45, fontsize=12, color=RECEPTOR_COLOR, weight='bold',
                style='italic', clip_on=False)

    # ── X 轴下排 sender 居中 ──
    sg = {}
    for i, cp in enumerate(cp_order):
        s = parse_cellpair(cp)[0]; sg.setdefault(s,[]).append(i)
    for s, idxs in sg.items():
        mid = (idxs[0]+idxs[-1])/2
        ax.plot([idxs[0]-0.35, idxs[-1]+0.35], [-0.17,-0.17],
                color='#afafaf', lw=1.0, transform=xt, clip_on=False)
        ax.text(mid, -0.31, s, transform=xt, ha='center', va='top',
                rotation=90, fontsize=12, color=LIGAND_COLOR, clip_on=False)

    ax.set_title(f'Differential ligand-receptor interactions: {g1} vs {g2}',
                 size=9, weight='bold', pad=8, loc='left')

    sh = []
    for v in [1,2,3]:
        _, sz = _dot_style(v, 0)
        sh.append(plt.scatter([],[],s=sz,c=[[*COLOR_G1,0.85]],
                  edgecolors='k',linewidths=0.6,label=f'-log10 p={v}'))
    ch = [mpatches.Patch(facecolor=COLOR_G1,   label=f'{g1} dominant'),
          mpatches.Patch(facecolor=COLOR_G2,   label=f'{g2} dominant'),
          mpatches.Patch(facecolor=COLOR_BOTH, label='Both sig.')]
    ax.legend(handles=sh+ch, loc='upper left', bbox_to_anchor=(1.01,0.50),
              frameon=False, fontsize=12, title='Legend',
              title_fontsize=12, handletextpad=0.8)

    plt.subplots_adjust(bottom=0.26, left=0.18, right=0.87, top=0.93)
    if save_path:
        save_pub_figure(fig, save_path, formats=('pdf', 'svg'), dpi=600)
    plt.show()
    return fig

# ── 预览 ──
fig4h_figs = {}
for g1, g2 in CONTRASTS_FIG4:
    print(f'\n===== {g1} vs {g2} =====')
    fig4h_figs[(g1,g2)] = fig4h_plot(group_lr, g1, g2, top_n=30, min_frac=0.05)


In [ ]:
# ============================================================
# 20b. Export Fig.4h as publication-ready vector files
# ============================================================
for g1, g2 in CONTRASTS_FIG4:
    base_path = FIG_DIR / f'fig4h_{g1}_vs_{g2}'
    print(f'Saving {g1} vs {g2} -> {base_path}')
    fig = fig4h_plot(group_lr, g1, g2, top_n=30, min_frac=0.05, save_path=base_path)
    if fig is not None:
        plt.close(fig)
print('Done. Fig.4h files saved to:', FIG_DIR)


In [ ]:
# ============================================================
# 21. Chord plot — 组特异 LRI（Fig.4d,e 风格）
# ============================================================
from matplotlib.path import Path as MPath
import matplotlib.patches as mpatches

CT_COLORS = CELLTYPE_COLORS.copy()

def _ct_color(ct):
    for k, v in CT_COLORS.items():
        if k.lower() in str(ct).lower(): return v
    return '#BAB0AC'

def _bezier_chord(ax, t1_mid, t2_mid, w1, w2, color, R=1.0):
    """Draw bezier chord between two points on circle."""
    p0 = R * np.array([np.cos(t1_mid - w1/2), np.sin(t1_mid - w1/2)])
    p1 = R * np.array([np.cos(t1_mid + w1/2), np.sin(t1_mid + w1/2)])
    p3 = R * np.array([np.cos(t2_mid - w2/2), np.sin(t2_mid - w2/2)])
    p2 = R * np.array([np.cos(t2_mid + w2/2), np.sin(t2_mid + w2/2)])
    verts = [p0, p0*0.0, p3*0.0, p3, p2, p2*0.0, p1*0.0, p1, p0]
    codes = [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.LINETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.CLOSEPOLY]
    ax.add_patch(mpatches.PathPatch(MPath(verts, codes),
                 facecolor=color, alpha=0.38, edgecolor='none', zorder=2))

def chord_diagram(mat, labels, title='', save_path=None):
    """mat[i,j] = n_LRI from sender i -> receiver j."""
    n = len(labels)
    totals = mat.sum(axis=1) + mat.sum(axis=0)
    total_all = totals.sum()
    if total_all == 0:
        print(f'  Empty matrix for {title}'); return

    gap = 2*np.pi*0.025
    avail = 2*np.pi - n*gap
    arcs = avail * totals / total_all
    starts = np.zeros(n)
    pos = np.pi/2
    for i in range(n):
        starts[i] = pos; pos += arcs[i] + gap

    fig, ax = plt.subplots(figsize=(6.6, 6.6), subplot_kw=dict(aspect='equal'))
    R = 1.0

    # Draw arcs
    for i in range(n):
        theta = np.linspace(starts[i], starts[i]+arcs[i], 60)
        x, y = R*np.cos(theta), R*np.sin(theta)
        ax.plot(x, y, lw=8, color=_ct_color(labels[i]), solid_capstyle='butt')
        mid = starts[i] + arcs[i]/2
        lr = 1.18*R
        ha = 'left' if np.cos(mid) >= 0 else 'right'
        rot = np.degrees(mid) - (0 if np.cos(mid) >= 0 else 180)
        ax.text(lr*np.cos(mid), lr*np.sin(mid), labels[i],
                ha=ha, va='center', fontsize=12, rotation=rot, weight='bold')

    # Draw chords
    offsets_out = np.zeros(n)
    offsets_in  = np.zeros(n)
    for i in range(n):
        for j in range(n):
            if mat[i,j] == 0: continue
            w_i = arcs[i] * mat[i,j] / max(totals[i], 1)
            w_j = arcs[j] * mat[i,j] / max(totals[j], 1)
            t_i = starts[i] + offsets_out[i] + w_i/2
            t_j = starts[j] + offsets_in[j]  + w_j/2
            offsets_out[i] += w_i
            offsets_in[j]  += w_j
            _bezier_chord(ax, t_i, t_j, w_i, w_j, _ct_color(labels[i]), R)

    ax.set_xlim(-1.55, 1.55); ax.set_ylim(-1.55, 1.55)
    ax.axis('off')
    ax.set_title(title, fontsize=12, weight='bold', pad=8, loc='left')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    if save_path:
        save_pub_figure(fig, save_path, formats=('pdf', 'svg'), dpi=600)
    plt.show()

# ── 每个对比：组特异 LRI 弦图 ──
for g1, g2 in CONTRASTS_FIG4:
    sig1 = set(all_full[(all_full['group']==g1)&(all_full['is_sig']==1)]
               [['cellpair','lr_pair']].apply(tuple, axis=1))
    sig2 = set(all_full[(all_full['group']==g2)&(all_full['is_sig']==1)]
               [['cellpair','lr_pair']].apply(tuple, axis=1))
    only1 = sig1 - sig2
    only2 = sig2 - sig1

    for label, only_set, grp in [(g1, only1, g1), (g2, only2, g2)]:
        rows = all_full[(all_full['group']==grp)&(all_full['is_sig']==1)].copy()
        rows['key'] = list(zip(rows['cellpair'], rows['lr_pair']))
        rows = rows[rows['key'].isin(only_set)]
        if rows.empty:
            print(f'  No {label}-specific LRI for {g1} vs {g2}'); continue
        cts = sorted(set(rows['sender'].tolist() + rows['receiver'].tolist()))
        ct_idx = {c:i for i,c in enumerate(cts)}
        mat = np.zeros((len(cts), len(cts)))
        for _, r in rows.iterrows():
            si, ri = ct_idx.get(r['sender']), ct_idx.get(r['receiver'])
            if si is not None and ri is not None:
                mat[si, ri] += 1
        chord_diagram(mat, cts,
                      title=f'{label}-specific LRI ({g1} vs {g2})',
                      save_path=FIG_DIR / f'chord_{label}_specific_{g1}_vs_{g2}.svg')


In [ ]:
# ============================================================
# 22. Key LR expression validation - violin plots (Fig.4f,g)
# ============================================================
gene_table = None
gene_map_path = DB_PATH / 'gene_table.csv'
if gene_map_path.exists():
    gene_table = pd.read_csv(gene_map_path)

hgnc_to_ensembl = {}
if gene_table is not None:
    for col_ens in ['ensembl', 'Ensembl', 'ensembl_id', 'gene_id']:
        if col_ens in gene_table.columns: break
    else: col_ens = None
    for col_name in ['gene_name', 'hgnc_symbol', 'name']:
        if col_name in gene_table.columns: break
    else: col_name = None
    if col_ens and col_name:
        for _, r in gene_table.iterrows():
            ens = str(r[col_ens]).strip()
            name = str(r[col_name]).strip().upper()
            if ens.startswith('ENSG') and name:
                hgnc_to_ensembl[name] = ens
        print(f'HGNC->Ensembl map: {len(hgnc_to_ensembl)} entries')

for col in ['gene_name', 'gene_short_name', 'Symbol', 'symbol']:
    if col in adata.var.columns:
        for ens_id, name in zip(adata.var_names, adata.var[col]):
            name = str(name).strip().upper()
            if name and name != 'NAN':
                hgnc_to_ensembl[name] = str(ens_id)
        print(f'Extended from adata.var["{col}"]: {len(hgnc_to_ensembl)} entries')
        break


def find_ensembl(symbol):
    s = str(symbol).strip().upper().replace('_HUMAN','')
    if s in hgnc_to_ensembl:
        eid = hgnc_to_ensembl[s]
        if eid in adata.var_names: return eid
    if s in adata.var_names: return s
    return None


if not lr_diff.empty:
    top_val = lr_diff.sort_values('pval').head(12).copy()
elif not pair_diff.empty:
    top_val = pair_diff.sort_values('pval').head(12).copy()
else:
    top_val = pd.DataFrame()
    print('No differential results.')

if not top_val.empty and 'ligand_gene' in top_val.columns:
    import scipy.sparse as sp_sparse

    def _get_expr_vector(gene_ens, mask):
        if gene_ens is None or mask.sum() == 0:
            return np.array([])
        x = adata[mask, gene_ens].X
        if sp_sparse.issparse(x):
            return np.asarray(x.todense()).flatten()
        return np.asarray(x).flatten()

    obs_group = normalize_group_series(adata.obs['diagnosis_group'].astype(str)).values
    obs_group = pd.Series(obs_group, index=adata.obs.index)
    groups_present = ordered_groups(obs_group)
    pal = group_palette(groups_present)
    n_saved = 0

    for _, r in top_val.iterrows():
        lig_ens = find_ensembl(r['ligand_gene'])
        rec_ens = find_ensembl(r['receptor_gene'])
        lr_name = r.get('lr_label', f"{r['ligand_gene']}-{r['receptor_gene']}")
        sender = str(r['sender'])
        receiver = str(r['receiver'])
        panels = []

        if lig_ens:
            frames = []
            for g in groups_present:
                mask = (obs_group.eq(g) & adata.obs['ccc_celltype'].astype(str).eq(sender)).values
                vals = _get_expr_vector(lig_ens, mask)
                if len(vals) > 0:
                    frames.append(pd.DataFrame({'group': g, 'expression': vals}))
            if frames:
                panels.append((f'Ligand: {r["ligand_gene"]}\nin {sender}', pd.concat(frames, ignore_index=True)))

        if rec_ens:
            frames = []
            for g in groups_present:
                mask = (obs_group.eq(g) & adata.obs['ccc_celltype'].astype(str).eq(receiver)).values
                vals = _get_expr_vector(rec_ens, mask)
                if len(vals) > 0:
                    frames.append(pd.DataFrame({'group': g, 'expression': vals}))
            if frames:
                panels.append((f'Receptor: {r["receptor_gene"]}\nin {receiver}', pd.concat(frames, ignore_index=True)))

        if not panels:
            print(f'Skip {lr_name}: no mapped expression data to plot.')
            continue

        fig_w = min(FIG_DOUBLE_WIDTH, max(FIG_SINGLE_WIDTH, 3.4 * len(panels)))
        fig, axes = plt.subplots(1, len(panels), figsize=(max(fig_w, 4.8 * len(panels)), 4.6), sharey=False, squeeze=False)
        axes = axes.ravel()
        for ax, (title, vdf) in zip(axes, panels):
            order = [g for g in groups_present if g in set(vdf['group'])]
            vdf['group'] = pd.Categorical(vdf['group'], categories=order, ordered=True)
            sns.violinplot(data=vdf, x='group', y='expression', order=order,
                           palette={g: pal[g] for g in order}, inner='quartile', linewidth=0.855,
                           cut=0, saturation=0.95, ax=ax)
            ax.set_title(title, loc='left')
            ax.set_ylabel('Log-normalized expression')
            ax.set_xlabel('')
            ax.tick_params(axis='x', rotation=0)
            clean_axes(ax, grid=True, grid_axis='y')

        fig.suptitle(f'{sender} -> {receiver} | {lr_name}', fontsize=12, fontweight='bold', y=1.02)
        save_pub_figure(fig, FIG_DIR / f'violin_{safe_filename(lr_name, 70)}')
        plt.show()
        n_saved += 1

    print(f'Done. Saved {n_saved} violin plot(s) to:', FIG_DIR)
else:
    print('Skipped (no mapping or no results).')


In [ ]:
# ============================================================
# 23. Pathway enrichment for differential LR genes (Fig.4i)
# ============================================================
import re


def extract_gene_symbols(name):
    s = str(name).replace('_HUMAN','')
    tokens = re.findall(r'[A-Za-z][A-Za-z0-9]{1,12}', s)
    stop = {'HUMAN','RECEPTOR','COMPLEX','UNKNOWN','BY','PROTEIN'}
    return sorted({t.upper() for t in tokens if t.upper() not in stop and len(t)>=2})


def collect_lr_genes(df, top_n=200):
    if df is None or df.empty: return []
    sub = df.sort_values('pval').head(top_n)
    genes = []
    for col in ['ligand_gene','receptor_gene']:
        if col not in sub.columns: continue
        for x in sub[col].dropna().astype(str):
            genes.extend(extract_gene_symbols(x))
    return sorted(set(genes))


def run_enrichr(gene_list, label, libraries=None):
    if libraries is None:
        libraries = ['GO_Biological_Process_2023','KEGG_2021_Human','Reactome_2022']
    out_file = ENRICH_DIR / f'{label}_genes.txt'
    out_file.write_text('\n'.join(gene_list), encoding='utf-8')
    if len(gene_list) < 5:
        print(f'  {label}: only {len(gene_list)} genes, skipped'); return pd.DataFrame()
    try:
        import gseapy as gp
    except ImportError:
        print(f'  gseapy not installed. Gene list saved: {out_file}'); return pd.DataFrame()
    all_res = []
    for lib in libraries:
        try:
            enr = gp.enrichr(gene_list=gene_list, gene_sets=lib,
                             organism='human', outdir=None, cutoff=1.0)
            res = enr.results.copy(); res['library'] = lib; res['label'] = label
            all_res.append(res)
        except Exception as e:
            print(f'  Enrichr failed {label}/{lib}: {e}')
    if not all_res: return pd.DataFrame()
    out = pd.concat(all_res, ignore_index=True)
    out.to_csv(ENRICH_DIR / f'{label}_enrichr.tsv', sep='\t', index=False)
    return out


enrich_all = []
if not lr_diff.empty:
    for contrast in lr_diff['contrast'].dropna().unique():
        sub = lr_diff[lr_diff['contrast']==contrast]
        up = sub[sub['log2fc_group1_vs_group2'] > 0]
        genes_up = collect_lr_genes(up, 200)
        print(f'{contrast} UP: {len(genes_up)} genes')
        res = run_enrichr(genes_up, f'{contrast}_UP')
        if not res.empty: enrich_all.append(res)
        dn = sub[sub['log2fc_group1_vs_group2'] < 0]
        genes_dn = collect_lr_genes(dn, 200)
        print(f'{contrast} DN: {len(genes_dn)} genes')
        res = run_enrichr(genes_dn, f'{contrast}_DN')
        if not res.empty: enrich_all.append(res)

enrichment_results = pd.concat(enrich_all, ignore_index=True) if enrich_all else pd.DataFrame()
if not enrichment_results.empty:
    enrichment_results.to_csv(ENRICH_DIR / 'all_enrichment_results.tsv', sep='\t', index=False)

if not enrichment_results.empty:
    p_col = 'Adjusted P-value' if 'Adjusted P-value' in enrichment_results.columns else 'P-value'
    term_col = 'Term' if 'Term' in enrichment_results.columns else enrichment_results.columns[0]
    enr = enrichment_results.copy()
    enr['neglog10_adjusted_p'] = -np.log10(pd.to_numeric(enr[p_col], errors='coerce').clip(lower=1e-300))
    enr['term_short'] = enr[term_col].astype(str).str.replace(r' \(GO.*\)$','',regex=True).str[:78]

    plot_enr = enr.sort_values(p_col).groupby('label', as_index=False).head(6).copy()
    term_order = plot_enr.groupby('term_short')['neglog10_adjusted_p'].max().sort_values().index.tolist()
    plot_enr['term_short'] = pd.Categorical(plot_enr['term_short'], categories=term_order, ordered=True)
    label_palette = categorical_palette(plot_enr['label'].dropna().unique())

    fig_h = max(3.8, min(9.0, 1.0 + 0.22 * len(term_order)))
    fig, ax = plt.subplots(figsize=(FIG_DOUBLE_WIDTH, fig_h))
    sns.scatterplot(
        data=plot_enr,
        x='neglog10_adjusted_p',
        y='term_short',
        hue='label',
        size='neglog10_adjusted_p',
        sizes=(24, 190),
        palette=label_palette,
        edgecolor='white',
        linewidth=0.655,
        alpha=0.95,
        ax=ax,
        legend='brief',
    )
    ax.set_title('Pathway enrichment of differential LR genes', loc='left')
    ax.set_xlabel('-log10 adjusted p-value')
    ax.set_ylabel('')
    wrap_axis_labels(ax, 'y', width=52)
    clean_axes(ax, grid=True, grid_axis='x')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=12, borderaxespad=0.4)
    fig.subplots_adjust(right=0.75)
    save_pub_figure(fig, FIG_DIR / 'pathway_enrichment_dotplot')
    plt.show()
else:
    print('No enrichment results. Gene lists saved to:', ENRICH_DIR)


In [ ]:
# ============================================================
# 24. Summary table — 每个对比的 top 信号通路和关键分子（用 p-value）
# ============================================================
summary_rows = []

if not lr_diff.empty:
    for contrast in lr_diff['contrast'].dropna().unique():
        sub = lr_diff[lr_diff['contrast']==contrast].sort_values('pval')

        # 用 p < 0.05（不做 FDR 校正）
        n_sig = len(sub[sub['pval'] < 0.05])

        top5 = sub.head(5)
        top_lig = top5['ligand_gene'].dropna().unique()[:5].tolist()
        top_rec = top5['receptor_gene'].dropna().unique()[:5].tolist()
        top_cp = top5[['sender','receiver']].drop_duplicates().head(3)
        top_cp_str = '; '.join(top_cp['sender'] + '→' + top_cp['receiver'])

        # top pathways — 去除重复 term
        top_pw = ''
        if not enrichment_results.empty:
            pw = enrichment_results[enrichment_results['label'].str.startswith(contrast)]
            p_col = 'Adjusted P-value' if 'Adjusted P-value' in pw.columns else 'P-value'
            term_col = 'Term' if 'Term' in pw.columns else pw.columns[0]
            if not pw.empty:
                top_pw = '; '.join(
                    pw.sort_values(p_col)
                    .drop_duplicates(subset=[term_col])   # 去重
                    .head(5)[term_col].astype(str).str[:60]
                )

        summary_rows.append({
            'contrast':       contrast,
            'n_sig_LR_p05':   n_sig,
            'top_ligands':    ', '.join(top_lig),
            'top_receptors':  ', '.join(top_rec),
            'top_cellpairs':  top_cp_str,
            'top_pathways':   top_pw,
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(TABLE_DIR / 'biological_summary_per_contrast.tsv', sep='\t', index=False)
display(summary)
print('Saved:', TABLE_DIR / 'biological_summary_per_contrast.tsv')


In [ ]:
# ============================================================
# 25. UMAP - diagnosis groups in one multi-panel figure
# ============================================================
from matplotlib.lines import Line2D

if 'X_umap' not in adata.obsm:
    print('Computing UMAP...')
    adata_plot = adata.copy()
    sc.pp.normalize_total(adata_plot, target_sum=1e4)
    sc.pp.log1p(adata_plot)
    sc.pp.highly_variable_genes(adata_plot, n_top_genes=2000)
    sc.pp.pca(adata_plot, use_highly_variable=True)
    sc.pp.neighbors(adata_plot)
    sc.tl.umap(adata_plot)
    adata.obsm['X_umap'] = adata_plot.obsm['X_umap']

coords = adata.obsm['X_umap']
UMAP_COLOR_KEY = 'major_celltype' if 'major_celltype' in adata.obs.columns else 'ccc_celltype'
ct_all = adata.obs[UMAP_COLOR_KEY].astype(str)
grp_all = normalize_group_series(adata.obs['diagnosis_group'].astype(str))
celltypes = sorted(ct_all.unique())
palette = categorical_palette(celltypes, fixed=CELLTYPE_COLORS)

legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=palette[c],
           markeredgecolor='none', markersize=8, label=c)
    for c in celltypes
]

groups = [g for g in ordered_groups(grp_all) if int((grp_all == g).sum()) > 0]
fig_w = min(7.2, max(5.8, 1.45 * len(groups)))
fig, axes = plt.subplots(1, len(groups), figsize=(max(fig_w, 4.2 * len(groups)), 3.4), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

x_pad = 0.03 * (coords[:, 0].max() - coords[:, 0].min())
y_pad = 0.03 * (coords[:, 1].max() - coords[:, 1].min())
x_min, x_max = coords[:, 0].min() - x_pad, coords[:, 0].max() + x_pad
y_min, y_max = coords[:, 1].min() - y_pad, coords[:, 1].max() + y_pad

rng = np.random.default_rng(7)
plot_order = rng.permutation(coords.shape[0])

for ax, grp in zip(axes, groups):
    grp_mask = (grp_all == grp).values
    bg = plot_order[~grp_mask[plot_order]]
    fg = plot_order[grp_mask[plot_order]]

    ax.scatter(coords[bg, 0], coords[bg, 1], s=1.4, color='#D9D9D9', alpha=0.22,
               linewidths=0, rasterized=True)
    for ct in celltypes:
        idx = fg[ct_all.iloc[fg].eq(ct).values]
        if idx.size == 0:
            continue
        ax.scatter(coords[idx, 0], coords[idx, 1], s=2.2, color=palette[ct], alpha=0.88,
                   linewidths=0, rasterized=True)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(grp, loc='left')
    ax.set_xlabel('UMAP 1')
    if ax is axes[0]:
        ax.set_ylabel('UMAP 2')
    else:
        ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.text(0.02, 0.98, f'n = {int(grp_mask.sum()):,}', transform=ax.transAxes,
            fontsize=12, va='top', color='#303030')

axes[-1].legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.02, 0.5),
                frameon=False, fontsize=12, title=UMAP_COLOR_KEY, title_fontsize=12,
                markerscale=1.4, handletextpad=0.6, borderaxespad=0.4)
fig.suptitle(f'UMAP by diagnosis group, colored by {UMAP_COLOR_KEY}', fontsize=12, fontweight='bold', y=1.04)
fig.subplots_adjust(wspace=0.24, right=0.80, top=0.84, bottom=0.18)
save_pub_figure(fig, FIG_DIR / 'umap_five_groups')
plt.show()


In [ ]:
# ============================================================
# 25. UMAP - one publication-ready panel per diagnosis group
# ============================================================
from matplotlib.lines import Line2D

if 'X_umap' not in adata.obsm:
    print('Computing UMAP...')
    adata_plot = adata.copy()
    sc.pp.normalize_total(adata_plot, target_sum=1e4)
    sc.pp.log1p(adata_plot)
    sc.pp.highly_variable_genes(adata_plot, n_top_genes=2000)
    sc.pp.pca(adata_plot, use_highly_variable=True)
    sc.pp.neighbors(adata_plot)
    sc.tl.umap(adata_plot)
    adata.obsm['X_umap'] = adata_plot.obsm['X_umap']

coords = adata.obsm['X_umap']
UMAP_COLOR_KEY = 'major_celltype' if 'major_celltype' in adata.obs.columns else 'ccc_celltype'
ct_all = adata.obs[UMAP_COLOR_KEY].astype(str)
grp_all = normalize_group_series(adata.obs['diagnosis_group'].astype(str))
celltypes = sorted(ct_all.unique())
palette = categorical_palette(celltypes, fixed=CELLTYPE_COLORS)

legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=palette[c],
           markeredgecolor='none', markersize=8, label=c)
    for c in celltypes
]

x_pad = 0.03 * (coords[:, 0].max() - coords[:, 0].min())
y_pad = 0.03 * (coords[:, 1].max() - coords[:, 1].min())
x_min, x_max = coords[:, 0].min() - x_pad, coords[:, 0].max() + x_pad
y_min, y_max = coords[:, 1].min() - y_pad, coords[:, 1].max() + y_pad

groups = [g for g in ordered_groups(grp_all) if int((grp_all == g).sum()) > 0]
rng = np.random.default_rng(7)
plot_order = rng.permutation(coords.shape[0])

for grp in groups:
    grp_mask = (grp_all == grp).values
    bg = plot_order[~grp_mask[plot_order]]
    fg = plot_order[grp_mask[plot_order]]
    fig, ax = plt.subplots(figsize=(3.35, 3.05))

    ax.scatter(coords[bg, 0], coords[bg, 1], s=1.4, color='#D9D9D9', alpha=0.22,
               linewidths=0, rasterized=True)
    for ct in celltypes:
        idx = fg[ct_all.iloc[fg].eq(ct).values]
        if idx.size == 0:
            continue
        ax.scatter(coords[idx, 0], coords[idx, 1], s=2.3, color=palette[ct], alpha=0.88,
                   linewidths=0, rasterized=True, label=ct)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'UMAP - {grp}', loc='left')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.text(0.02, 0.98, f'n = {int(grp_mask.sum()):,}', transform=ax.transAxes,
            fontsize=12, va='top', color='#303030')

    ax.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.02, 0.5),
              frameon=False, fontsize=12, title=UMAP_COLOR_KEY, title_fontsize=12,
              markerscale=1.4, handletextpad=0.6, borderaxespad=0.4)
    fig.subplots_adjust(right=0.68)
    save_pub_figure(fig, FIG_DIR / f'umap_{safe_filename(grp)}')
    plt.show()
    print(f'Saved UMAP files for {grp}')
